In [1]:
# -*- coding: utf-8 -*-
"""探测违规类型文件格式"""
import os
import pandas as pd

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

for fname in ["违规目标四类_公司年度类型.csv", "违规标签_公司年度.csv",
              "Desc_ViolationTypes.csv", "ViolationType_stratification.csv"]:
    path = os.path.join(BASE, fname)
    if os.path.exists(path):
        df = pd.read_csv(path, encoding='utf-8-sig')
        print(f"\n{'='*60}")
        print(f"文件：{fname}")
        print(f"Shape: {df.shape}")
        print(f"列名：{df.columns.tolist()}")
        print(f"前 5 行：")
        print(df.head())
    else:
        print(f"\n未找到：{fname}")


文件：违规目标四类_公司年度类型.csv
Shape: (9392, 9)
列名：['ViolationID', 'Stkcd', 'year', 'ViolationTypeID', 'ViolationTypeName', 'IsViolated', 'DisposalDate', 'DeclareDate', 'year_source']
前 5 行：
   ViolationID  Stkcd  year ViolationTypeID ViolationTypeName IsViolated  \
0     40121966      4  2022           P2503       虚假记载(误导性陈述)          Y   
1     40127455      4  2022           P2503       虚假记载(误导性陈述)          Y   
2     40138124      4  2022           P2503       虚假记载(误导性陈述)          Y   
3      4013493      7  2012           P2503       虚假记载(误导性陈述)          Y   
4      4013493      7  2013           P2503       虚假记载(误导性陈述)          Y   

  DisposalDate DeclareDate    year_source  
0   2022-08-03  2022-08-05  ViolationYear  
1   2022-11-15  2022-11-15  ViolationYear  
2   2023-06-14  2023-06-14  ViolationYear  
3   2014-06-16  2014-06-18  ViolationYear  
4   2014-06-16  2014-06-18  ViolationYear  

文件：违规标签_公司年度.csv
Shape: (5008, 7)
列名：['Stkcd', 'year', 'fraud_types', 'fraud_count', 'Fraud', 'f

In [2]:
# -*- coding: utf-8 -*-
"""
P0-1：狭义舞弊标签稳健性检验（修正版）
- 宽口径：P2501 + P2502 + P2503 + P2506（原主实验）
- 窄口径：仅 P2501 + P2502（虚构利润 + 虚列资产）
- 从明细表 违规目标四类_公司年度类型.csv 提取窄口径标签
输出：
  Narrow_Fraud_Sample_Check.csv
  Narrow_Fraud_Comparison.csv
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

# ============================================================
# 1. 读取建模数据
# ============================================================
df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_未隔离.csv"), encoding='utf-8-sig')
print(f"原始样本：{len(df)}，宽口径舞弊率：{df['Fraud'].mean():.4f}")

# ============================================================
# 2. 读取违规明细，构建窄口径标签
# ============================================================
viol = pd.read_csv(os.path.join(BASE, "违规目标四类_公司年度类型.csv"), encoding='utf-8-sig')
print(f"\n违规明细 shape：{viol.shape}")
print(f"违规类型分布：")
print(viol['ViolationTypeID'].value_counts())

# ---------- Stkcd 归一化 ----------
def norm_stkcd(s):
    return s.astype(str).str.replace(r'\.0$', '', regex=True).str.strip().str.zfill(6)

viol['Stkcd'] = norm_stkcd(viol['Stkcd'])
df['Stkcd'] = norm_stkcd(df['Stkcd'])

# ---------- 筛选 P2501 + P2502 ----------
narrow = viol[viol['ViolationTypeID'].isin(['P2501', 'P2502'])]
narrow_keys = narrow[['Stkcd', 'year']].drop_duplicates()
narrow_keys['Fraud_narrow'] = 1

print(f"\n窄口径（P2501+P2502）firm-year 数：{len(narrow_keys)}")
print(f"  其中 P2501 only: {len(narrow[narrow['ViolationTypeID']=='P2501'][['Stkcd','year']].drop_duplicates())}")
print(f"  其中 P2502 only: {len(narrow[narrow['ViolationTypeID']=='P2502'][['Stkcd','year']].drop_duplicates())}")

# ---------- Merge 到建模数据 ----------
df = df.merge(narrow_keys, on=['Stkcd', 'year'], how='left')
df['Fraud_narrow'] = df['Fraud_narrow'].fillna(0).astype(int)

print(f"\n建模数据窄口径样本：{df['Fraud_narrow'].sum()}，"
      f"舞弊率：{df['Fraud_narrow'].mean():.4f}")

# ---------- 样本检查表 ----------
wide_only = int(((df['Fraud'] == 1) & (df['Fraud_narrow'] == 0)).sum())
narrow_only = int(((df['Fraud'] == 0) & (df['Fraud_narrow'] == 1)).sum())
both = int(((df['Fraud'] == 1) & (df['Fraud_narrow'] == 1)).sum())

check = pd.DataFrame({
    "Category": [
        "Wide fraud (P2501+P2502+P2503+P2506)",
        "Narrow fraud (P2501+P2502)",
        "Both wide and narrow",
        "Wide only (P2503/P2506 only)",
        "Narrow only (unexpected)",
        "Non-fraud (both = 0)"
    ],
    "Observations": [
        int(df['Fraud'].sum()),
        int(df['Fraud_narrow'].sum()),
        both, wide_only, narrow_only,
        int(((df['Fraud'] == 0) & (df['Fraud_narrow'] == 0)).sum())
    ]
})
check.to_csv(os.path.join(OUT, "Narrow_Fraud_Sample_Check.csv"),
             index=False, encoding='utf-8-sig')
print("\n样本检查：")
print(check.to_string(index=False))

# ============================================================
# 3. 特征列
# ============================================================
exclude_cols = ['Stkcd', 'year', 'Fraud', 'Fraud_narrow', 'ShortName',
                'IndustryName1', 'ViolationTypeID', 'DeclareDate', 'DisposalDate',
                'Enddate', 'set']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"\n最终特征数：{len(feature_cols)}")

# ============================================================
# 4. 模型参数
# ============================================================
params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f1, best_th = 0, 0.5
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th

def eval_model(sub_df, feature_list, label_col, label):
    train = sub_df[sub_df['year'] <= 2021]
    val   = sub_df[sub_df['year'] == 2022]
    test  = sub_df[sub_df['year'] >= 2023]

    if len(test) == 0 or test[label_col].sum() == 0 or len(val) == 0:
        print(f"  [{label}] 数据不足，跳过")
        return None

    X_train, y_train = train[feature_list], train[label_col].values
    X_val,   y_val   = val[feature_list],   val[label_col].values
    X_test,  y_test  = test[feature_list],  test[label_col].values

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    prob_val = model.predict_proba(X_val)[:, 1]
    best_th = find_best_threshold(y_val, prob_val)

    prob_test = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob_test)
    pr_auc = average_precision_score(y_test, prob_test)

    y_pred = (prob_test >= best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        'label': label,
        'n_train': len(train), 'n_test': len(test),
        'test_fraud_rate': round(y_test.mean(), 4),
        'AUC': round(auc, 4), 'PR_AUC': round(pr_auc, 4),
        'Recall': round(recall, 4), 'Precision': round(precision, 4),
        'Type_II_Error': round(type2, 4),
        'threshold': round(best_th, 3),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)
    }

# ============================================================
# 5. 两种标签对比
# ============================================================
print("\n训练 Wide label ...")
res_wide = eval_model(df, feature_cols, 'Fraud', 'Wide (P2501+P2502+P2503+P2506)')

print("训练 Narrow label ...")
res_narrow = eval_model(df, feature_cols, 'Fraud_narrow', 'Narrow (P2501+P2502)')

results = [r for r in [res_wide, res_narrow] if r is not None]
comparison = pd.DataFrame(results)
comparison.to_csv(os.path.join(OUT, "Narrow_Fraud_Comparison.csv"),
                  index=False, encoding='utf-8-sig')

print("\n窄口径 vs 宽口径对比：")
print(comparison.to_string(index=False))

print("\n完成！输出：")
print("  Narrow_Fraud_Sample_Check.csv")
print("  Narrow_Fraud_Comparison.csv")

原始样本：48328，宽口径舞弊率：0.0876

违规明细 shape：(9392, 9)
违规类型分布：
P2503    7562
P2501    1407
P2506     231
P2502     192
Name: ViolationTypeID, dtype: int64

窄口径（P2501+P2502）firm-year 数：991
  其中 P2501 only: 938
  其中 P2502 only: 132

建模数据窄口径样本：842，舞弊率：0.0174

样本检查：
                            Category  Observations
Wide fraud (P2501+P2502+P2503+P2506)          4233
          Narrow fraud (P2501+P2502)           842
                Both wide and narrow           842
        Wide only (P2503/P2506 only)          3391
            Narrow only (unexpected)             0
                Non-fraud (both = 0)         44095

最终特征数：89

训练 Wide label ...
训练 Narrow label ...

窄口径 vs 宽口径对比：
                         label  n_train  n_test  test_fraud_rate    AUC  PR_AUC  Recall  Precision  Type_II_Error  threshold  TP  FN   FP    TN
Wide (P2501+P2502+P2503+P2506)    31553   11237           0.0815 0.7668  0.2705  0.5229     0.2287         0.4771       0.67 479 437 1615  8706
          Narrow (P2501+P2502)    31

In [3]:
# -*- coding: utf-8 -*-
"""
P0-2：成本比敏感性
- GroupKFold OOF 和 TimeSplit Test，分别扫描 5:1, 10:1, 20:1
输出：
  Threshold_Cost_Ratios_GroupKFold.csv
  Threshold_Cost_Ratios_TimeSplit.csv
  Threshold_Cost_Ratios_Summary.csv
"""
import os
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT = BASE

def cost_analysis(y_true, y_prob, label, cost_ratios=[5, 10, 20]):
    thresholds = np.arange(0.02, 0.99, 0.01)
    rows = []
    for ratio in cost_ratios:
        best_cost = np.inf
        best_metrics = None
        for th in thresholds:
            y_pred = (y_prob >= th).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
            cost = ratio * fn + 1 * fp
            if cost < best_cost:
                best_cost = cost
                prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
                type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0
                type1 = fp / (fp + tn) if (fp + tn) > 0 else 0.0
                best_metrics = {
                    'dataset': label, 'cost_ratio_FN:FP': f'{ratio}:1',
                    'best_threshold': round(th, 3),
                    'Recall': round(rec, 4),
                    'Precision': round(prec, 4),
                    'F1': round(f1, 4),
                    'Type_I_Error': round(type1, 4),
                    'Type_II_Error': round(type2, 4),
                    'TP': int(tp), 'FP': int(fp),
                    'FN': int(fn), 'TN': int(tn),
                    'total_cost': int(best_cost)
                }
        if best_metrics:
            rows.append(best_metrics)
    return pd.DataFrame(rows)

# ---------- GroupKFold ----------
print("=" * 60)
print("GroupKFold OOF")
print("=" * 60)
df_gk = pd.read_csv(os.path.join(BASE, "Optuna_calibrated_predictions.csv"),
                     encoding='utf-8-sig')
y_true = df_gk['y_true'].values
prob_col = 'y_prob_calibrated' if 'y_prob_calibrated' in df_gk.columns else 'y_prob_raw'
y_prob = df_gk[prob_col].values

gk_df = cost_analysis(y_true, y_prob, 'GroupKFold')
gk_df.to_csv(os.path.join(OUT, "Threshold_Cost_Ratios_GroupKFold.csv"),
             index=False, encoding='utf-8-sig')
print(gk_df.to_string(index=False))

# ---------- TimeSplit ----------
print("\n" + "=" * 60)
print("TimeSplit Test")
print("=" * 60)
ts_path = os.path.join(BASE, "TimeSplit_test_predictions.csv")
if os.path.exists(ts_path):
    df_ts = pd.read_csv(ts_path, encoding='utf-8-sig')
    y_true_ts = df_ts['y_true'].values
    for col in ['prob_cal', 'y_prob_calibrated', 'prob_raw']:
        if col in df_ts.columns:
            y_prob_ts = df_ts[col].values
            print(f"使用概率列：{col}")
            break
    ts_df = cost_analysis(y_true_ts, y_prob_ts, 'TimeSplit Test')
    ts_df.to_csv(os.path.join(OUT, "Threshold_Cost_Ratios_TimeSplit.csv"),
                 index=False, encoding='utf-8-sig')
    print(ts_df.to_string(index=False))
    summary = pd.concat([gk_df, ts_df], ignore_index=True)
else:
    summary = gk_df

summary.to_csv(os.path.join(OUT, "Threshold_Cost_Ratios_Summary.csv"),
               index=False, encoding='utf-8-sig')
print("\n完成！输出：Threshold_Cost_Ratios_*.csv")

GroupKFold OOF
   dataset cost_ratio_FN:FP  best_threshold  Recall  Precision     F1  Type_I_Error  Type_II_Error   TP    FP   FN    TN  total_cost
GroupKFold              5:1            0.16  0.4954     0.2639 0.3444        0.1326         0.5046 2097  5849 2136 38246       16529
GroupKFold             10:1            0.10  0.6834     0.1981 0.3072        0.2655         0.3166 2893 11708 1340 32387       25108
GroupKFold             20:1            0.05  0.9003     0.1352 0.2352        0.5526         0.0997 3811 24369  422 19726       32809

TimeSplit Test
使用概率列：prob_cal
       dataset cost_ratio_FN:FP  best_threshold  Recall  Precision     F1  Type_I_Error  Type_II_Error  TP   FP  FN   TN  total_cost
TimeSplit Test              5:1            0.16  0.4814     0.2337 0.3147        0.1401         0.5186 441 1446 475 8875        3821
TimeSplit Test             10:1            0.13  0.5655     0.2151 0.3117        0.1831         0.4345 518 1890 398 8431        5870
TimeSplit Test         

In [4]:
# -*- coding: utf-8 -*-
"""
P0-3：多重共线性分析
输出：
  VIF_Financial_NonFinancial.csv
  Correlation_Matrix.csv
  High_Correlation_Pairs.csv
"""
import os
import numpy as np
import pandas as pd

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT = BASE

df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_未隔离.csv"),
                 encoding='utf-8-sig')
print(f"原始 shape: {df.shape}")

# ---------- MD&A 列（用于排除）----------
mda_cols = ['TextualSimilarity', 'PositiveVocabularyNum', 'NegativeVocabularyNum',
            'EmotionTone1', 'EmotionTone2', 'PosRatio', 'NegRatio',
            'SentLenAvg', 'SentLenStd', 'ComplexWordRatio', 'DigitDensity',
            'PuncDensity', 'TTR', 'Jaccard_prev', 'EditSim_prev',
            'TFIDF_Cosine_prev', 'DLUT_PosNum', 'DLUT_NegNum',
            'DLUT_PosRatio', 'DLUT_NegRatio', 'DLUT_PosIntensity',
            'DLUT_NegIntensity', 'DLUT_EmotionScore', 'DLUT_EmotionTone',
            'DLUT_NegAfterNeg', 'DLUT_PosAfterNeg']

exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set']
exclude_cols = [c for c in exclude_cols if c in df.columns]

fn_cols = []
for c in df.columns:
    if c in exclude_cols: continue
    if c in mda_cols: continue
    if df[c].dtype in ['int64', 'float64', 'int32', 'float32']:
        fn_cols.append(c)

print(f"\n财务+非财务特征数：{len(fn_cols)}")

# ---------- 缺失值与常数处理 ----------
X = df[fn_cols].copy()
nunique = X.nunique()
const_cols = nunique[nunique <= 1].index.tolist()
if const_cols:
    print(f"剔除常数/全缺失列 {len(const_cols)} 个")
    X = X.drop(columns=const_cols)
    fn_cols = [c for c in fn_cols if c not in const_cols]

X = X.fillna(X.median())

# ---------- 尝试 VIF（需要 statsmodels）----------
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    print("\n计算 VIF ...")
    vif_data = []
    for i, col in enumerate(X.columns):
        try:
            v = variance_inflation_factor(X.values, i)
        except Exception:
            v = np.nan
        vif_data.append({'feature': col, 'VIF': v})

    vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False).reset_index(drop=True)
    vif_df['VIF'] = vif_df['VIF'].round(3)
    vif_df['VIF_gt_10'] = (vif_df['VIF'] > 10).astype(int)
    vif_df['VIF_gt_5'] = (vif_df['VIF'] > 5).astype(int)

    vif_df.to_csv(os.path.join(OUT, "VIF_Financial_NonFinancial.csv"),
                  index=False, encoding='utf-8-sig')
    print(f"\nVIF > 10 的特征数：{vif_df['VIF_gt_10'].sum()} / {len(vif_df)}")
    print(f"VIF > 5  的特征数：{vif_df['VIF_gt_5'].sum()} / {len(vif_df)}")
    print("\nVIF Top 20：")
    print(vif_df.head(20).to_string(index=False))
except ImportError:
    print("\n⚠️ statsmodels 未安装，跳过 VIF。运行：pip install statsmodels")

# ---------- 相关矩阵 ----------
print("\n计算相关矩阵 ...")
corr = X.corr()
corr.to_csv(os.path.join(OUT, "Correlation_Matrix.csv"), encoding='utf-8-sig')

high_corr = []
cols = corr.columns.tolist()
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.8:
            high_corr.append({'feature_1': cols[i], 'feature_2': cols[j],
                              'correlation': round(r, 3)})

high_corr_df = pd.DataFrame(high_corr) if high_corr else \
    pd.DataFrame(columns=['feature_1', 'feature_2', 'correlation'])
if len(high_corr_df) > 0:
    high_corr_df = high_corr_df.reindex(
        high_corr_df['correlation'].abs().sort_values(ascending=False).index)

high_corr_df.to_csv(os.path.join(OUT, "High_Correlation_Pairs.csv"),
                    index=False, encoding='utf-8-sig')
print(f"\n|r| > 0.8 的特征对：{len(high_corr_df)}")
if len(high_corr_df) > 0:
    print(high_corr_df.head(20).to_string(index=False))

print("\n完成！输出：")
print("  VIF_Financial_NonFinancial.csv")
print("  Correlation_Matrix.csv")
print("  High_Correlation_Pairs.csv")

原始 shape: (48328, 88)

财务+非财务特征数：57

计算 VIF ...

VIF > 10 的特征数：10 / 57
VIF > 5  的特征数：15 / 57

VIF Top 20：
                feature      VIF  VIF_gt_10  VIF_gt_5
               F080501A 1431.486          1         1
               F082201B 1427.292          1         1
               F060301B  568.494          1         1
               F053401B  516.222          1         1
               F051301B  330.733          1         1
               F050201B  305.846          1         1
               F050101B  295.926          1         1
               F053301B   91.422          1         1
               F010201A   35.876          1         1
               F010101A   32.256          1         1
               F051701B    6.964          0         1
ChairmanHoldsharesRatio    5.914          0         1
               Mngmhldn    5.654          0         1
               F050301B    5.554          0         1
      LargestHolderRate    5.033          0         1
     ContrshrProportion    4.2

In [5]:
# -*- coding: utf-8 -*-
"""
Winsorize 稳健性检验
- 对 57 个财务+非财务特征做 1%/99% winsorize（边界在 train 上计算）
- 重跑时间外推主模型，比较 AUC / PR-AUC / Recall / Type II
输出：
  Winsorize_Comparison.csv
  Winsorize_Bounds.csv
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

# ============================================================
# 1. 读取建模数据
# ============================================================
df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_未隔离.csv"), encoding='utf-8-sig')
print(f"原始 shape: {df.shape}")

# ============================================================
# 2. 定义 MD&A 列（winsorize 只对财务+非财务做）
# ============================================================
mda_cols = ['TextualSimilarity', 'PositiveVocabularyNum', 'NegativeVocabularyNum',
            'EmotionTone1', 'EmotionTone2', 'PosRatio', 'NegRatio',
            'SentLenAvg', 'SentLenStd', 'ComplexWordRatio', 'DigitDensity',
            'PuncDensity', 'TTR', 'Jaccard_prev', 'EditSim_prev',
            'TFIDF_Cosine_prev', 'DLUT_PosNum', 'DLUT_NegNum',
            'DLUT_PosRatio', 'DLUT_NegRatio', 'DLUT_PosIntensity',
            'DLUT_NegIntensity', 'DLUT_EmotionScore', 'DLUT_EmotionTone',
            'DLUT_NegAfterNeg', 'DLUT_PosAfterNeg']

exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set']
exclude_cols = [c for c in exclude_cols if c in df.columns]

feature_cols = [c for c in df.columns if c not in exclude_cols]

# one-hot object 列
obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]

# winsorize 目标列：财务+非财务（不含 MD&A、不含 one-hot 后的 dummy）
winsor_cols = []
for c in feature_cols:
    if c in mda_cols: continue
    if df[c].dtype in ['int64', 'float64', 'int32', 'float32']:
        # 排除 0/1 二值变量
        nunique = df[c].nunique()
        if nunique > 5:
            winsor_cols.append(c)

print(f"winsorize 目标列：{len(winsor_cols)} 个")

# ============================================================
# 3. 划分 train/val/test
# ============================================================
train = df[df['year'] <= 2021].copy()
val   = df[df['year'] == 2022].copy()
test  = df[df['year'] >= 2023].copy()

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

# ============================================================
# 4. 在 train 上计算 1%/99% 分位数边界
# ============================================================
bounds = {}
for c in winsor_cols:
    lo = train[c].quantile(0.01)
    hi = train[c].quantile(0.99)
    bounds[c] = (lo, hi)

bounds_df = pd.DataFrame([
    {'feature': c, 'lower_1pct': bounds[c][0], 'upper_99pct': bounds[c][1]}
    for c in winsor_cols
])
bounds_df.to_csv(os.path.join(OUT, "Winsorize_Bounds.csv"), index=False, encoding='utf-8-sig')

# ============================================================
# 5. 应用 winsorize（clip 到 train 边界）
# ============================================================
def apply_winsorize(sub_df):
    sub_df = sub_df.copy()
    for c in winsor_cols:
        lo, hi = bounds[c]
        sub_df[c] = sub_df[c].clip(lower=lo, upper=hi)
    return sub_df

train_w = apply_winsorize(train)
val_w   = apply_winsorize(val)
test_w  = apply_winsorize(test)

# ============================================================
# 6. 模型参数
# ============================================================
params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f1, best_th = 0, 0.5
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th

def eval_set(sub_train, sub_val, sub_test, feature_list, label):
    X_train, y_train = sub_train[feature_list], sub_train['Fraud'].values
    X_val,   y_val   = sub_val[feature_list],   sub_val['Fraud'].values
    X_test,  y_test  = sub_test[feature_list],  sub_test['Fraud'].values

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    prob_val = model.predict_proba(X_val)[:, 1]
    best_th = find_best_threshold(y_val, prob_val)

    prob_test = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob_test)
    pr_auc = average_precision_score(y_test, prob_test)

    y_pred = (prob_test >= best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        'label': label,
        'n_train': len(sub_train), 'n_test': len(sub_test),
        'AUC': round(auc, 4), 'PR_AUC': round(pr_auc, 4),
        'Recall': round(recall, 4), 'Precision': round(precision, 4),
        'Type_II_Error': round(type2, 4),
        'threshold': round(best_th, 3),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)
    }

# ============================================================
# 7. 两种方案对比
# ============================================================
print("\n训练 Original ...")
res_orig = eval_set(train, val, test, feature_cols, 'Original')

print("训练 Winsorized ...")
res_wins = eval_set(train_w, val_w, test_w, feature_cols, 'Winsorized (1%/99%)')

comparison = pd.DataFrame([res_orig, res_wins])
comparison.to_csv(os.path.join(OUT, "Winsorize_Comparison.csv"),
                  index=False, encoding='utf-8-sig')

print("\nWinsorize 对比：")
print(comparison.to_string(index=False))

auc_diff = res_wins['AUC'] - res_orig['AUC']
print(f"\nAUC 差异（Winsorized - Original）：{auc_diff:+.4f}")
if abs(auc_diff) < 0.005:
    print("→ 差异 < 0.005，winsorize 不影响主结论")
else:
    print("→ 差异 > 0.005，需在论文中讨论")

print("\n完成！输出：")
print("  Winsorize_Comparison.csv")
print("  Winsorize_Bounds.csv")

原始 shape: (48328, 88)
winsorize 目标列：50 个
Train: 31553, Val: 5538, Test: 11237

训练 Original ...
训练 Winsorized ...

Winsorize 对比：
              label  n_train  n_test    AUC  PR_AUC  Recall  Precision  Type_II_Error  threshold  TP  FN   FP   TN
           Original    31553   11237 0.7668  0.2705  0.5229     0.2287         0.4771       0.67 479 437 1615 8706
Winsorized (1%/99%)    31553   11237 0.7663  0.2699  0.5066     0.2359         0.4934       0.68 464 452 1503 8818

AUC 差异（Winsorized - Original）：-0.0005
→ 差异 < 0.005，winsorize 不影响主结论

完成！输出：
  Winsorize_Comparison.csv
  Winsorize_Bounds.csv


In [6]:
# -*- coding: utf-8 -*-
"""
行业未匹配样本稳健性
- 对比：全样本 vs 仅保留有行业信息的样本
- 重跑时间外推主模型，比较 AUC / PR-AUC
输出：
  Industry_Unmatched_Comparison.csv
  Industry_Unmatched_Sample_Check.csv
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

# ============================================================
# 1. 读取带行业信息的建模数据
# ============================================================
df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_行业.csv"), encoding='utf-8-sig')
print(f"总样本：{len(df)}")
print(f"行业已匹配：{df['IndustryName'].notna().sum()}")
print(f"行业未匹配：{df['IndustryName'].isna().sum()}")

# ============================================================
# 2. 样本检查
# ============================================================
matched = df[df['IndustryName'].notna()]
unmatched = df[df['IndustryName'].isna()]

check = pd.DataFrame({
    "Category": ["Total", "Industry matched", "Industry unmatched",
                 "Unmatched fraud", "Unmatched non-fraud",
                 "Unmatched fraud rate (%)"],
    "Value": [
        len(df),
        len(matched),
        len(unmatched),
        int(unmatched['Fraud'].sum()),
        int((unmatched['Fraud'] == 0).sum()),
        round(unmatched['Fraud'].mean() * 100, 4)
    ]
})
check.to_csv(os.path.join(OUT, "Industry_Unmatched_Sample_Check.csv"),
             index=False, encoding='utf-8-sig')
print("\n样本检查：")
print(check.to_string(index=False))

# ============================================================
# 3. 特征列
# ============================================================
exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1',
                'IndustryName', 'ViolationTypeID', 'DeclareDate', 'DisposalDate',
                'Enddate', 'set']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"\n最终特征数：{len(feature_cols)}")

# ============================================================
# 4. 模型参数
# ============================================================
params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f1, best_th = 0, 0.5
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th

def eval_model(sub_df, feature_list, label):
    train = sub_df[sub_df['year'] <= 2021]
    val   = sub_df[sub_df['year'] == 2022]
    test  = sub_df[sub_df['year'] >= 2023]

    if len(test) == 0 or test['Fraud'].sum() == 0 or len(val) == 0:
        return None

    X_train, y_train = train[feature_list], train['Fraud'].values
    X_val,   y_val   = val[feature_list],   val['Fraud'].values
    X_test,  y_test  = test[feature_list],  test['Fraud'].values

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    prob_val = model.predict_proba(X_val)[:, 1]
    best_th = find_best_threshold(y_val, prob_val)

    prob_test = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob_test)
    pr_auc = average_precision_score(y_test, prob_test)

    y_pred = (prob_test >= best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        'sample': label,
        'n_train': len(train), 'n_test': len(test),
        'test_fraud_rate': round(y_test.mean(), 4),
        'AUC': round(auc, 4), 'PR_AUC': round(pr_auc, 4),
        'Recall': round(recall, 4), 'Precision': round(precision, 4),
        'Type_II_Error': round(type2, 4),
        'threshold': round(best_th, 3),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)
    }

# ============================================================
# 5. 两种样本对比
# ============================================================
print("\n训练 Full sample ...")
res_full = eval_model(df, feature_cols, 'Full sample (48,328)')

print("训练 Matched-only ...")
res_matched = eval_model(df[df['IndustryName'].notna()], feature_cols,
                          'Industry matched only')

results = [r for r in [res_full, res_matched] if r is not None]
comparison = pd.DataFrame(results)
comparison.to_csv(os.path.join(OUT, "Industry_Unmatched_Comparison.csv"),
                  index=False, encoding='utf-8-sig')

print("\n行业未匹配稳健性对比：")
print(comparison.to_string(index=False))

if len(results) == 2:
    auc_diff = results[1]['AUC'] - results[0]['AUC']
    print(f"\nAUC 差异（Matched - Full）：{auc_diff:+.4f}")
    if abs(auc_diff) < 0.005:
        print("→ 差异 < 0.005，未匹配样本不影响主结论")
    else:
        print("→ 差异 > 0.005，需在论文中讨论")

print("\n完成！输出：")
print("  Industry_Unmatched_Comparison.csv")
print("  Industry_Unmatched_Sample_Check.csv")

总样本：48328
行业已匹配：41565
行业未匹配：6763

样本检查：
                Category      Value
                   Total 48328.0000
        Industry matched 41565.0000
      Industry unmatched  6763.0000
         Unmatched fraud    92.0000
     Unmatched non-fraud  6671.0000
Unmatched fraud rate (%)     1.3603

最终特征数：89

训练 Full sample ...
训练 Matched-only ...

行业未匹配稳健性对比：
               sample  n_train  n_test  test_fraud_rate    AUC  PR_AUC  Recall  Precision  Type_II_Error  threshold  TP  FN   FP   TN
 Full sample (48,328)    31553   11237           0.0815 0.7668  0.2705  0.5229     0.2287         0.4771       0.67 479 437 1615 8706
Industry matched only    26018   10528           0.0831 0.7440  0.2514  0.5166     0.2184         0.4834       0.62 452 423 1618 8035

AUC 差异（Matched - Full）：-0.0228
→ 差异 > 0.005，需在论文中讨论

完成！输出：
  Industry_Unmatched_Comparison.csv
  Industry_Unmatched_Sample_Check.csv


In [7]:
# -*- coding: utf-8 -*-
"""
非财务变量缺失率查询
- 检查现有建模数据中非财务特征的缺失率
- 检查 CSMAR 非财务指标目录中是否有额外的治理/审计变量
输出：
  NonFinancial_Missing_Rates.csv
  NonFinancial_Extra_Available.csv
"""
import os
import pandas as pd

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载"
OUT  = os.path.join(BASE, "合并结果")

# ============================================================
# 1. 建模数据中非财务特征的缺失率
# ============================================================
df = pd.read_csv(os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv"), encoding='utf-8-sig')
print(f"建模数据 shape: {df.shape}")

# 已知非财务特征（从你的特征列表）
nonfin_features = [
    'InternationalBig4', 'TotalAuditFee', 'ContrshrProportion', 'Mngmhldn',
    'Boardsize', 'IndDirectorRatio', 'SupervisorSize', 'Y0301b', 'Y0501b',
    'ChairmanHoldsharesRatio', 'ManagerHoldsharesRatio', 'Y1001b',
    'LargestHolderRate', 'TopTenHoldersRate',
    'IsDisclosingEvaRep', 'IsValid', 'IsDeficiency', 'TypeAuditOpin'
]

print(f"\n已知非财务特征：{len(nonfin_features)} 个")
print(f"其中在建模数据中的：{len([c for c in nonfin_features if c in df.columns])} 个")

rows = []
for c in nonfin_features:
    if c in df.columns:
        rows.append({
            'feature': c,
            'dtype': str(df[c].dtype),
            'missing_count': int(df[c].isna().sum()),
            'missing_rate': round(df[c].isna().mean(), 4),
            'n_unique': int(df[c].nunique())
        })
    else:
        rows.append({
            'feature': c,
            'dtype': 'NOT_IN_DATA',
            'missing_count': None,
            'missing_rate': None,
            'n_unique': None
        })

missing_df = pd.DataFrame(rows).sort_values('missing_rate', ascending=False)
missing_df.to_csv(os.path.join(OUT, "NonFinancial_Missing_Rates.csv"),
                  index=False, encoding='utf-8-sig')

print("\n非财务特征缺失率：")
print(missing_df.to_string(index=False))

# ============================================================
# 2. 检查 CSMAR 原始非财务目录中是否有额外变量
# ============================================================
print("\n" + "=" * 60)
print("CSMAR 非财务目录检查")
print("=" * 60)

nonfin_dir = os.path.join(BASE, "非财务指标", "保留数据")
if os.path.exists(nonfin_dir):
    files = os.listdir(nonfin_dir)
    print(f"目录文件：{files}")

    extra_info = []
    for f in files:
        path = os.path.join(nonfin_dir, f)
        if not os.path.isfile(path):
            continue
        try:
            if f.endswith('.xlsx'):
                sub = pd.read_excel(path)
            elif f.endswith('.csv'):
                sub = pd.read_csv(path, encoding='utf-8-sig')
            else:
                continue

            print(f"\n--- {f} ---")
            print(f"Shape: {sub.shape}")
            print(f"列名（前 30）：{sub.columns.tolist()[:30]}")

            # 检查是否包含 R1 点名的变量
            target_keywords = ['audit committee', '审计委员会',
                                'auditor tenure', '审计任期',
                                'institutional', '机构持股',
                                'CFO', '财务总监',
                                'turnover', '变更', '离职',
                                'compensation', '薪酬',
                                'board size', '董事会规模',
                                'board independence', '独立董事']
            matched = []
            for c in sub.columns:
                c_str = str(c).lower()
                for kw in target_keywords:
                    if kw.lower() in c_str:
                        matched.append(c)
                        break
            if matched:
                print(f"  含目标关键词的列：{matched}")
                extra_info.append({'file': f, 'columns': ', '.join(matched)})
            else:
                print(f"  未发现 R1 点名的变量")
        except Exception as e:
            print(f"读取 {f} 失败：{e}")

    if extra_info:
        extra_df = pd.DataFrame(extra_info)
        extra_df.to_csv(os.path.join(OUT, "NonFinancial_Extra_Available.csv"),
                        index=False, encoding='utf-8-sig')
        print(f"\n发现 {len(extra_info)} 个文件含 R1 点名的变量关键词")
    else:
        print("\n未发现 R1 点名的额外变量，需要 Response 中用数据限制解释")
else:
    print(f"非财务目录不存在：{nonfin_dir}")

print("\n完成！输出：")
print("  NonFinancial_Missing_Rates.csv")
print("  NonFinancial_Extra_Available.csv（如果有额外变量）")

建模数据 shape: (48328, 88)

已知非财务特征：18 个
其中在建模数据中的：18 个

非财务特征缺失率：
                feature   dtype  missing_count  missing_rate  n_unique
 ManagerHoldsharesRatio float64          12575        0.2602      3914
ChairmanHoldsharesRatio float64          12034        0.2490      4647
               Mngmhldn float64          11859        0.2454     19845
           IsDeficiency float64          11737        0.2429         2
                IsValid float64          11739        0.2429         2
     ContrshrProportion float64          11731        0.2427      6253
                 Y1001b float64          11379        0.2355         2
       IndDirectorRatio float64          10826        0.2240        40
              Boardsize float64          10823        0.2239        17
         SupervisorSize float64          10823        0.2239        13
     IsDisclosingEvaRep float64          10638        0.2201         2
          TotalAuditFee float64          10362        0.2144      2173
             

D:\aca\lib\site-packages\openpyxl\styles\stylesheet.py:221: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")



--- AIQ_FinAuditOpinY.xlsx ---
Shape: (45796, 5)
列名（前 30）：['Symbol', 'EndDate', 'TypeAuditOpin', 'InternationalBig4', 'TotalAuditFee']
  未发现 R1 点名的变量

--- BDT_ManaGovAbil.xlsx ---
Shape: (45193, 8)
列名（前 30）：['Symbol', 'ShortName', 'Enddate', 'ContrshrProportion', 'Mngmhldn', 'Boardsize', 'IndDirectorRatio', 'SupervisorSize']
  未发现 R1 点名的变量

--- CG_Ybasic.xlsx ---
Shape: (45797, 7)
列名（前 30）：['Stkcd', 'Reptdt', 'Y0301b', 'Y0501b', 'ChairmanHoldsharesRatio', 'ManagerHoldsharesRatio', 'Y1001b']
  未发现 R1 点名的变量

--- EN_EquityNatureAll.xlsx ---
Shape: (45797, 5)
列名（前 30）：['Symbol', 'ShortName', 'EndDate', 'LargestHolderRate', 'TopTenHoldersRate']
  未发现 R1 点名的变量

--- FIN_Audit.xlsx ---
Shape: (45798, 4)
列名（前 30）：['Stkcd', 'Stknme', 'Accper', 'Audittyp']
  未发现 R1 点名的变量

--- IC_EvaluationRepInfo.xlsx ---
Shape: (45292, 5)
列名（前 30）：['Symbol', 'EndDate', 'IsDisclosingEvaRep', 'IsValid', 'IsDeficiency']
  未发现 R1 点名的变量

未发现 R1 点名的额外变量，需要 Response 中用数据限制解释

完成！输出：
  NonFinancial_Missing_Rates.csv
  

In [8]:
from sklearn.isotonic import IsotonicRegression

def calibrate_and_evaluate(model, X_val, y_val, X_test, y_test):
    prob_val = model.predict_proba(X_val)[:, 1]
    prob_test_raw = model.predict_proba(X_test)[:, 1]
    
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(prob_val, y_val)
    prob_test_cal = iso.predict(prob_test_raw)
    
    return prob_test_raw, prob_test_cal

In [9]:
# -*- coding: utf-8 -*-
"""
ST 敏感性检验（加 Isotonic 校准）
- Full sample vs Exclude ST/*ST
- 统一报告校准后 AUC / PR-AUC / Recall / Precision / Type II
输出：
  ST_Sensitivity_Comparison_Calibrated.csv
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score
from sklearn.isotonic import IsotonicRegression

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE
industry_path = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\行业数据.xlsx"

# ============================================================
# 1. 读取建模数据 + 提取 ShortName 标记 ST
# ============================================================
df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_未隔离.csv"), encoding='utf-8-sig')

ind = pd.read_excel(industry_path, sheet_name=0, dtype={'Symbol': str})
ind = ind[ind['Symbol'].astype(str).str.match(r'^\d{6}$', na=False)].copy()

def norm_stkcd(s):
    return s.astype(str).str.replace(r'\.0$', '', regex=True).str.strip().str.zfill(6)

ind['Symbol'] = norm_stkcd(ind['Symbol'])
df['Stkcd'] = norm_stkcd(df['Stkcd'])
ind['year'] = pd.to_datetime(ind['EndDate'], errors='coerce').dt.year
ind = ind.dropna(subset=['year'])
ind['year'] = ind['year'].astype(int)
ind = ind.sort_values(['Symbol', 'year'])
ind_short = ind.drop_duplicates(subset=['Symbol', 'year'], keep='last')[['Symbol', 'year', 'ShortName']]

df = df.merge(ind_short, left_on=['Stkcd', 'year'], right_on=['Symbol', 'year'], how='left')
if 'Symbol' in df.columns:
    df = df.drop(columns=['Symbol'])

df['is_ST'] = df['ShortName'].astype(str).str.contains('ST', na=False)
print(f"ST/*ST 样本：{df['is_ST'].sum()} / {len(df)} = {df['is_ST'].mean():.2%}")

# ============================================================
# 2. 特征列
# ============================================================
exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set', 'is_ST']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"特征数：{len(feature_cols)}")

# ============================================================
# 3. 模型参数
# ============================================================
params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f1, best_th = 0, 0.5
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th

def eval_with_calibration(sub_df, feature_list, label):
    """训练 → 验证集拟合 Isotonic → 测试集评估（校准后）"""
    train = sub_df[sub_df['year'] <= 2021]
    val   = sub_df[sub_df['year'] == 2022]
    test  = sub_df[sub_df['year'] >= 2023]

    if len(test) == 0 or test['Fraud'].sum() == 0 or len(val) == 0:
        return None

    X_train, y_train = train[feature_list], train['Fraud'].values
    X_val,   y_val   = val[feature_list],   val['Fraud'].values
    X_test,  y_test  = test[feature_list],  test['Fraud'].values

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    # 验证集概率 → 拟合 Isotonic
    prob_val_raw = model.predict_proba(X_val)[:, 1]
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(prob_val_raw, y_val)

    # 测试集：原始 + 校准
    prob_test_raw = model.predict_proba(X_test)[:, 1]
    prob_test_cal = iso.predict(prob_test_raw)

    auc_raw = roc_auc_score(y_test, prob_test_raw)
    auc_cal = roc_auc_score(y_test, prob_test_cal)
    pr_auc_cal = average_precision_score(y_test, prob_test_cal)

    # 阈值在验证集校准概率上选
    prob_val_cal = iso.predict(prob_val_raw)
    best_th = find_best_threshold(y_val, prob_val_cal)

    y_pred = (prob_test_cal >= best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        'sample': label,
        'n_train': len(train), 'n_test': len(test),
        'test_fraud_rate': round(y_test.mean(), 4),
        'AUC_raw': round(auc_raw, 4),
        'AUC_calibrated': round(auc_cal, 4),
        'PR_AUC_calibrated': round(pr_auc_cal, 4),
        'Recall': round(recall, 4), 'Precision': round(precision, 4),
        'Type_II_Error': round(type2, 4),
        'threshold': round(best_th, 3),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)
    }

# ============================================================
# 4. 两种样本对比
# ============================================================
print("\n训练 Full sample ...")
res_full = eval_with_calibration(df, feature_cols, 'Full sample (includes ST)')

print("训练 Exclude ST ...")
res_no_st = eval_with_calibration(df[~df['is_ST']], feature_cols, 'Exclude ST/*ST')

results = [r for r in [res_full, res_no_st] if r is not None]
comparison = pd.DataFrame(results)
comparison.to_csv(os.path.join(OUT, "ST_Sensitivity_Comparison_Calibrated.csv"),
                  index=False, encoding='utf-8-sig')

print("\nST 敏感性（校准后）：")
print(comparison.to_string(index=False))

print("\n完成！输出：ST_Sensitivity_Comparison_Calibrated.csv")

ST/*ST 样本：1170 / 48328 = 2.42%
特征数：89

训练 Full sample ...
训练 Exclude ST ...

ST 敏感性（校准后）：
                   sample  n_train  n_test  test_fraud_rate  AUC_raw  AUC_calibrated  PR_AUC_calibrated  Recall  Precision  Type_II_Error  threshold  TP  FN   FP   TN
Full sample (includes ST)    31553   11237           0.0815   0.7668          0.7642             0.2489  0.5142     0.2307         0.4858       0.17 471 445 1571 8750
           Exclude ST/*ST    30739   11005           0.0751   0.7479          0.7465             0.2036  0.4673     0.2065         0.5327       0.17 386 440 1483 8696

完成！输出：ST_Sensitivity_Comparison_Calibrated.csv


In [10]:
# -*- coding: utf-8 -*-
"""
行业未匹配稳健性（加 Isotonic 校准）
- Full sample vs Industry-matched-only
输出：
  Industry_Unmatched_Comparison_Calibrated.csv
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score
from sklearn.isotonic import IsotonicRegression

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_行业.csv"), encoding='utf-8-sig')
print(f"总样本：{len(df)}")
print(f"行业已匹配：{df['IndustryName'].notna().sum()}")
print(f"行业未匹配：{df['IndustryName'].isna().sum()}")

unmatched = df[df['IndustryName'].isna()]
print(f"未匹配样本中舞弊：{unmatched['Fraud'].sum()}，非舞弊：{(unmatched['Fraud']==0).sum()}")

# 特征列
exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1', 'IndustryName',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"特征数：{len(feature_cols)}")

params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f1, best_th = 0, 0.5
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th

def eval_with_calibration(sub_df, feature_list, label):
    train = sub_df[sub_df['year'] <= 2021]
    val   = sub_df[sub_df['year'] == 2022]
    test  = sub_df[sub_df['year'] >= 2023]

    if len(test) == 0 or test['Fraud'].sum() == 0 or len(val) == 0:
        return None

    X_train, y_train = train[feature_list], train['Fraud'].values
    X_val,   y_val   = val[feature_list],   val['Fraud'].values
    X_test,  y_test  = test[feature_list],  test['Fraud'].values

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    prob_val_raw = model.predict_proba(X_val)[:, 1]
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(prob_val_raw, y_val)

    prob_test_raw = model.predict_proba(X_test)[:, 1]
    prob_test_cal = iso.predict(prob_test_raw)

    auc_raw = roc_auc_score(y_test, prob_test_raw)
    auc_cal = roc_auc_score(y_test, prob_test_cal)
    pr_auc_cal = average_precision_score(y_test, prob_test_cal)

    prob_val_cal = iso.predict(prob_val_raw)
    best_th = find_best_threshold(y_val, prob_val_cal)

    y_pred = (prob_test_cal >= best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        'sample': label,
        'n_train': len(train), 'n_test': len(test),
        'test_fraud_rate': round(y_test.mean(), 4),
        'AUC_raw': round(auc_raw, 4),
        'AUC_calibrated': round(auc_cal, 4),
        'PR_AUC_calibrated': round(pr_auc_cal, 4),
        'Recall': round(recall, 4), 'Precision': round(precision, 4),
        'Type_II_Error': round(type2, 4),
        'threshold': round(best_th, 3),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)
    }

print("\n训练 Full sample ...")
res_full = eval_with_calibration(df, feature_cols, 'Full sample (48,328)')

print("训练 Matched-only ...")
res_matched = eval_with_calibration(df[df['IndustryName'].notna()],
                                     feature_cols, 'Industry matched only')

results = [r for r in [res_full, res_matched] if r is not None]
comparison = pd.DataFrame(results)
comparison.to_csv(os.path.join(OUT, "Industry_Unmatched_Comparison_Calibrated.csv"),
                  index=False, encoding='utf-8-sig')

print("\n行业未匹配（校准后）：")
print(comparison.to_string(index=False))

print("\n完成！输出：Industry_Unmatched_Comparison_Calibrated.csv")

总样本：48328
行业已匹配：41565
行业未匹配：6763
未匹配样本中舞弊：92，非舞弊：6671
特征数：89

训练 Full sample ...
训练 Matched-only ...

行业未匹配（校准后）：
               sample  n_train  n_test  test_fraud_rate  AUC_raw  AUC_calibrated  PR_AUC_calibrated  Recall  Precision  Type_II_Error  threshold  TP  FN   FP   TN
 Full sample (48,328)    31553   11237           0.0815   0.7668          0.7642             0.2489  0.5142     0.2307         0.4858       0.17 471 445 1571 8750
Industry matched only    26018   10528           0.0831   0.7440          0.7437             0.2305  0.5097     0.2195         0.4903       0.14 446 429 1586 8067

完成！输出：Industry_Unmatched_Comparison_Calibrated.csv


In [11]:
# -*- coding: utf-8 -*-
"""
Winsorize 稳健性（加 Isotonic 校准）
- 边界在 train 上计算，clip 到 val/test
- 报告校准后 AUC
输出：
  Winsorize_Comparison_Calibrated.csv
  Winsorize_Bounds.csv
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score
from sklearn.isotonic import IsotonicRegression

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_未隔离.csv"), encoding='utf-8-sig')

mda_cols = ['TextualSimilarity', 'PositiveVocabularyNum', 'NegativeVocabularyNum',
            'EmotionTone1', 'EmotionTone2', 'PosRatio', 'NegRatio',
            'SentLenAvg', 'SentLenStd', 'ComplexWordRatio', 'DigitDensity',
            'PuncDensity', 'TTR', 'Jaccard_prev', 'EditSim_prev',
            'TFIDF_Cosine_prev', 'DLUT_PosNum', 'DLUT_NegNum',
            'DLUT_PosRatio', 'DLUT_NegRatio', 'DLUT_PosIntensity',
            'DLUT_NegIntensity', 'DLUT_EmotionScore', 'DLUT_EmotionTone',
            'DLUT_NegAfterNeg', 'DLUT_PosAfterNeg']

exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]

# winsorize 目标列
winsor_cols = []
for c in feature_cols:
    if c in mda_cols: continue
    if df[c].dtype in ['int64', 'float64', 'int32', 'float32']:
        if df[c].nunique() > 5:
            winsor_cols.append(c)
print(f"winsorize 目标列：{len(winsor_cols)}")

# 划分
train = df[df['year'] <= 2021].copy()
val   = df[df['year'] == 2022].copy()
test  = df[df['year'] >= 2023].copy()

# 在 train 上计算 1%/99% 边界
bounds = {}
for c in winsor_cols:
    bounds[c] = (train[c].quantile(0.01), train[c].quantile(0.99))

pd.DataFrame([
    {'feature': c, 'lower_1pct': bounds[c][0], 'upper_99pct': bounds[c][1]}
    for c in winsor_cols
]).to_csv(os.path.join(OUT, "Winsorize_Bounds.csv"), index=False, encoding='utf-8-sig')

def apply_winsorize(sub_df):
    sub_df = sub_df.copy()
    for c in winsor_cols:
        lo, hi = bounds[c]
        sub_df[c] = sub_df[c].clip(lower=lo, upper=hi)
    return sub_df

train_w, val_w, test_w = apply_winsorize(train), apply_winsorize(val), apply_winsorize(test)

params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f1, best_th = 0, 0.5
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th

def eval_with_calibration(sub_train, sub_val, sub_test, feature_list, label):
    X_train, y_train = sub_train[feature_list], sub_train['Fraud'].values
    X_val,   y_val   = sub_val[feature_list],   sub_val['Fraud'].values
    X_test,  y_test  = sub_test[feature_list],  sub_test['Fraud'].values

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    prob_val_raw = model.predict_proba(X_val)[:, 1]
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(prob_val_raw, y_val)

    prob_test_raw = model.predict_proba(X_test)[:, 1]
    prob_test_cal = iso.predict(prob_test_raw)

    auc_raw = roc_auc_score(y_test, prob_test_raw)
    auc_cal = roc_auc_score(y_test, prob_test_cal)
    pr_auc_cal = average_precision_score(y_test, prob_test_cal)

    prob_val_cal = iso.predict(prob_val_raw)
    best_th = find_best_threshold(y_val, prob_val_cal)

    y_pred = (prob_test_cal >= best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        'label': label,
        'n_train': len(sub_train), 'n_test': len(sub_test),
        'AUC_raw': round(auc_raw, 4),
        'AUC_calibrated': round(auc_cal, 4),
        'PR_AUC_calibrated': round(pr_auc_cal, 4),
        'Recall': round(recall, 4), 'Precision': round(precision, 4),
        'Type_II_Error': round(type2, 4),
        'threshold': round(best_th, 3),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)
    }

print("\n训练 Original ...")
res_orig = eval_with_calibration(train, val, test, feature_cols, 'Original')

print("训练 Winsorized ...")
res_wins = eval_with_calibration(train_w, val_w, test_w, feature_cols, 'Winsorized (1%/99%)')

comparison = pd.DataFrame([res_orig, res_wins])
comparison.to_csv(os.path.join(OUT, "Winsorize_Comparison_Calibrated.csv"),
                  index=False, encoding='utf-8-sig')

print("\nWinsorize 对比（校准后）：")
print(comparison.to_string(index=False))

print("\n完成！输出：Winsorize_Comparison_Calibrated.csv")

winsorize 目标列：50

训练 Original ...
训练 Winsorized ...

Winsorize 对比（校准后）：
              label  n_train  n_test  AUC_raw  AUC_calibrated  PR_AUC_calibrated  Recall  Precision  Type_II_Error  threshold  TP  FN   FP   TN
           Original    31553   11237   0.7668          0.7642             0.2489  0.5142     0.2307         0.4858       0.17 471 445 1571 8750
Winsorized (1%/99%)    31553   11237   0.7663          0.7646             0.2518  0.4913     0.2371         0.5087       0.19 450 466 1448 8873

完成！输出：Winsorize_Comparison_Calibrated.csv


In [12]:
# -*- coding: utf-8 -*-
"""
行业分层 + 行业 FE（加 Isotonic 校准）
- Full / Manufacturing / Non-Manufacturing
- Full + Industry FE
输出：
  Industry_Stratification_Calibrated.csv
  Industry_FE_Comparison_Calibrated.csv
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score
from sklearn.isotonic import IsotonicRegression

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_行业.csv"), encoding='utf-8-sig')
df = df[df['IndustryName'].notna()].copy()
print(f"行业分析样本：{len(df)}")

df['is_manufacturing'] = df['IndustryName'].str.contains('制造业', na=False).astype(int)
print(f"制造业：{df['is_manufacturing'].sum()}，非制造业：{(df['is_manufacturing']==0).sum()}")

# 行业门类映射
def map_to_category(name):
    if pd.isna(name): return 'Unknown'
    if '制造业' in name: return 'Manufacturing'
    if '信息传输' in name or '软件' in name: return 'IT'
    if '批发' in name or '零售' in name: return 'Wholesale_Retail'
    if '房地产' in name: return 'Real_Estate'
    if '建筑' in name: return 'Construction'
    if '金融' in name: return 'Finance'
    if '电力' in name or '热力' in name or '燃气' in name or '水' in name: return 'Utilities'
    if '交通' in name or '运输' in name or '仓储' in name or '邮政' in name: return 'Transport'
    if '农' in name or '林' in name or '牧' in name or '渔' in name: return 'Agriculture'
    if '采矿' in name: return 'Mining'
    if '住宿' in name or '餐饮' in name: return 'Hospitality'
    if '租赁' in name or '商务服务' in name: return 'Leasing_Business'
    if '科学研究' in name or '技术服务' in name: return 'Science_Tech'
    if '水利' in name or '环境' in name or '公共设施' in name: return 'Water_Environment'
    if '教育' in name: return 'Education'
    if '卫生' in name or '社会工作' in name: return 'Health'
    if '文化' in name or '体育' in name or '娱乐' in name: return 'Culture_Sports'
    if '综合' in name: return 'Conglomerate'
    return 'Other'

df['IndustryCategory'] = df['IndustryName'].apply(map_to_category)

# 特征列
exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1', 'IndustryName',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set',
                'is_manufacturing', 'IndustryCategory']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"特征数：{len(feature_cols)}")

params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f1, best_th = 0, 0.5
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th

def eval_with_calibration(sub_df, feature_list, label):
    train = sub_df[sub_df['year'] <= 2021]
    val   = sub_df[sub_df['year'] == 2022]
    test  = sub_df[sub_df['year'] >= 2023]

    if len(test) == 0 or test['Fraud'].sum() == 0 or len(val) == 0:
        return None

    X_train, y_train = train[feature_list], train['Fraud'].values
    X_val,   y_val   = val[feature_list],   val['Fraud'].values
    X_test,  y_test  = test[feature_list],  test['Fraud'].values

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    prob_val_raw = model.predict_proba(X_val)[:, 1]
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(prob_val_raw, y_val)

    prob_test_raw = model.predict_proba(X_test)[:, 1]
    prob_test_cal = iso.predict(prob_test_raw)

    auc_raw = roc_auc_score(y_test, prob_test_raw)
    auc_cal = roc_auc_score(y_test, prob_test_cal)
    pr_auc_cal = average_precision_score(y_test, prob_test_cal)

    prob_val_cal = iso.predict(prob_val_raw)
    best_th = find_best_threshold(y_val, prob_val_cal)

    y_pred = (prob_test_cal >= best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        'sample': label,
        'n_train': len(train), 'n_test': len(test),
        'test_fraud_rate': round(y_test.mean(), 4),
        'AUC_raw': round(auc_raw, 4),
        'AUC_calibrated': round(auc_cal, 4),
        'PR_AUC_calibrated': round(pr_auc_cal, 4),
        'Recall': round(recall, 4), 'Precision': round(precision, 4),
        'Type_II_Error': round(type2, 4),
        'threshold': round(best_th, 3),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)
    }

# ============================================================
# 1. 制造业 vs 非制造业
# ============================================================
print("\n训练 Full sample ...")
res_full = eval_with_calibration(df, feature_cols, 'Full sample')

print("训练 Manufacturing ...")
res_mfg = eval_with_calibration(df[df['is_manufacturing'] == 1], feature_cols, 'Manufacturing')

print("训练 Non-Manufacturing ...")
res_non = eval_with_calibration(df[df['is_manufacturing'] == 0], feature_cols, 'Non-Manufacturing')

strat = pd.DataFrame([r for r in [res_full, res_mfg, res_non] if r is not None])
strat.to_csv(os.path.join(OUT, "Industry_Stratification_Calibrated.csv"),
             index=False, encoding='utf-8-sig')
print("\n行业分层（校准后）：")
print(strat.to_string(index=False))

# ============================================================
# 2. 行业固定效应
# ============================================================
print("\n训练 Full + Industry FE ...")
ind_dummies = pd.get_dummies(df['IndustryCategory'], prefix='IND')
df_fe = pd.concat([df, ind_dummies], axis=1)
fe_cols = feature_cols + list(ind_dummies.columns)
print(f"加入行业 FE 后特征数：{len(fe_cols)}")

res_fe = eval_with_calibration(df_fe, fe_cols, 'Full + Industry FE')

fe_compare = pd.DataFrame([r for r in [res_full, res_fe] if r is not None])
fe_compare.to_csv(os.path.join(OUT, "Industry_FE_Comparison_Calibrated.csv"),
                  index=False, encoding='utf-8-sig')
print("\n行业 FE 前后对比（校准后）：")
print(fe_compare.to_string(index=False))

print("\n完成！输出：")
print("  Industry_Stratification_Calibrated.csv")
print("  Industry_FE_Comparison_Calibrated.csv")

行业分析样本：41565
制造业：21724，非制造业：19841
特征数：89

训练 Full sample ...
训练 Manufacturing ...
训练 Non-Manufacturing ...

行业分层（校准后）：
           sample  n_train  n_test  test_fraud_rate  AUC_raw  AUC_calibrated  PR_AUC_calibrated  Recall  Precision  Type_II_Error  threshold  TP  FN   FP   TN
      Full sample    26018   10528           0.0831   0.7440          0.7437             0.2305  0.5097     0.2195         0.4903       0.14 446 429 1586 8067
    Manufacturing    13206    5815           0.0739   0.7449          0.7425             0.2074  0.4721     0.2278         0.5279       0.17 203 227  688 4697
Non-Manufacturing    12812    4713           0.0944   0.7274          0.7267             0.2437  0.6090     0.1886         0.3910       0.14 271 174 1166 3102

训练 Full + Industry FE ...
加入行业 FE 后特征数：107

行业 FE 前后对比（校准后）：
            sample  n_train  n_test  test_fraud_rate  AUC_raw  AUC_calibrated  PR_AUC_calibrated  Recall  Precision  Type_II_Error  threshold  TP  FN   FP   TN
       Full sample    2

In [13]:
import pandas as pd
df = pd.read_csv(r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_未隔离.csv")

exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

mda_cols = ['TextualSimilarity', 'PositiveVocabularyNum', 'NegativeVocabularyNum',
            'EmotionTone1', 'EmotionTone2', 'PosRatio', 'NegRatio',
            'SentLenAvg', 'SentLenStd', 'ComplexWordRatio', 'DigitDensity',
            'PuncDensity', 'TTR', 'Jaccard_prev', 'EditSim_prev',
            'TFIDF_Cosine_prev', 'DLUT_PosNum', 'DLUT_NegNum',
            'DLUT_PosRatio', 'DLUT_NegRatio', 'DLUT_PosIntensity',
            'DLUT_NegIntensity', 'DLUT_EmotionScore', 'DLUT_EmotionTone',
            'DLUT_NegAfterNeg', 'DLUT_PosAfterNeg']

fin_cols = [c for c in feature_cols if c.startswith('F0')]
nonfin_cols = [c for c in feature_cols if c not in fin_cols and c not in mda_cols]

print(f"特征总数：{len(feature_cols)}")
print(f"财务：{len(fin_cols)}")
print(f"非财务：{len(nonfin_cols)}")
print(f"  {nonfin_cols}")
print(f"MD&A：{len([c for c in feature_cols if c in mda_cols])}")

特征总数：84
财务：40
非财务：18
  ['TypeAuditOpin', 'InternationalBig4', 'TotalAuditFee', 'ContrshrProportion', 'Mngmhldn', 'Boardsize', 'IndDirectorRatio', 'SupervisorSize', 'Y0301b', 'Y0501b', 'ChairmanHoldsharesRatio', 'ManagerHoldsharesRatio', 'Y1001b', 'LargestHolderRate', 'TopTenHoldersRate', 'IsDisclosingEvaRep', 'IsValid', 'IsDeficiency']
MD&A：26


In [14]:
import pandas as pd
from sklearn.metrics import brier_score_loss

# 时间外推
df_ts = pd.read_csv(r"D:\...\合并结果\TimeSplit_test_predictions.csv")
brier_raw = brier_score_loss(df_ts['y_true'], df_ts['prob_raw'])
brier_cal = brier_score_loss(df_ts['y_true'], df_ts['prob_cal'])
print(f"TimeSplit: raw={brier_raw:.4f}, calibrated={brier_cal:.4f}")

# GroupKFold
df_gk = pd.read_csv(r"D:\...\合并结果\Optuna_calibrated_predictions.csv")
brier_raw_gk = brier_score_loss(df_gk['y_true'], df_gk['y_prob_raw'])
brier_cal_gk = brier_score_loss(df_gk['y_true'], df_gk['y_prob_calibrated'])
print(f"GroupKFold: raw={brier_raw_gk:.4f}, calibrated={brier_cal_gk:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\...\\合并结果\\TimeSplit_test_predictions.csv'

In [15]:
import pandas as pd
from sklearn.metrics import brier_score_loss

df_ts = pd.read_csv(r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\TimeSplit_test_predictions.csv")
brier_raw = brier_score_loss(df_ts['y_true'], df_ts['prob_raw'])
brier_cal = brier_score_loss(df_ts['y_true'], df_ts['prob_cal'])
print(f"TimeSplit: raw={brier_raw:.4f}, calibrated={brier_cal:.4f}")

df_gk = pd.read_csv(r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\Optuna_calibrated_predictions.csv")
brier_raw_gk = brier_score_loss(df_gk['y_true'], df_gk['y_prob_raw'])
brier_cal_gk = brier_score_loss(df_gk['y_true'], df_gk['y_prob_calibrated'])
print(f"GroupKFold: raw={brier_raw_gk:.4f}, calibrated={brier_cal_gk:.4f}")

TimeSplit: raw=0.2112, calibrated=0.0693
GroupKFold: raw=0.1617, calibrated=0.0704


In [2]:
# -*- coding: utf-8 -*-
"""
搜索 TimeSplit 相关文件的实际位置
"""
import os

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载"

# 递归搜索所有 TimeSplit 开头的文件
print("=" * 70)
print("搜索 TimeSplit 文件：")
print("=" * 70)

found = []
for root, dirs, files in os.walk(BASE):
    for f in files:
        if 'TimeSplit' in f or 'timesplit' in f.lower() or 'time_split' in f.lower():
            full_path = os.path.join(root, f)
            size = os.path.getsize(full_path) / 1024  # KB
            found.append((full_path, size))
            print(f"{full_path}  ({size:.1f} KB)")

if not found:
    print("未找到 TimeSplit 文件")

print("\n" + "=" * 70)
print("搜索所有 test_predictions 文件：")
print("=" * 70)

for root, dirs, files in os.walk(BASE):
    for f in files:
        if 'test_predictions' in f.lower() or 'test_pred' in f.lower():
            full_path = os.path.join(root, f)
            size = os.path.getsize(full_path) / 1024
            print(f"{full_path}  ({size:.1f} KB)")

print("\n" + "=" * 70)
print("搜索所有 Bootstrap_CI 文件：")
print("=" * 70)

for root, dirs, files in os.walk(BASE):
    for f in files:
        if 'Bootstrap' in f or 'bootstrap' in f.lower():
            full_path = os.path.join(root, f)
            size = os.path.getsize(full_path) / 1024
            print(f"{full_path}  ({size:.1f} KB)")

搜索 TimeSplit 文件：
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\PR_HighRecall_Threshold_Cost_TimeSplit.png  (135.9 KB)
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\Threshold_Cost_Ratios_TimeSplit.csv  (0.4 KB)
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\Threshold_Cost_TimeSplit.csv  (12.6 KB)
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\Threshold_Cost_TimeSplit.png  (134.4 KB)
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\TimeSplit_best_params.csv  (0.3 KB)
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\TimeSplit_Bootstrap_CI.csv  (0.6 KB)
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\TimeSplit_metrics.csv  (0.8 KB)
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\TimeSplit_test_predictions.csv  (461.0 KB)
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\TimeSplit_val_predictions.csv  (227.2 KB)

搜索所有 test_predictions 文件：
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\lgb_test_predictions.csv  (1990.1 KB)
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\TimeSplit_test_predictions.csv  (461.0 KB)

搜索所有 Bootstrap_CI 文件：
D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\TimeSplit_Bootstrap_CI.csv  (0.6 KB)


In [3]:
# -*- coding: utf-8 -*-
"""
生成 Table 3：完整 7-8 行
- 6 行 GroupKFold OOF
- 1 行 TimeSplit 主结果
"""
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score, brier_score_loss

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

# ============================================================
# 1. 读取 GroupKFold OOF
# ============================================================
oof = pd.read_csv(os.path.join(BASE, "对比_OOF_predictions.csv"), encoding='utf-8-sig')
lasso = pd.read_csv(os.path.join(BASE, "LassoLR_OOF.csv"), encoding='utf-8-sig')
optuna = pd.read_csv(os.path.join(BASE, "Optuna_calibrated_predictions.csv"), encoding='utf-8-sig')
stack = pd.read_csv(os.path.join(BASE, "Stacking_OOF.csv"), encoding='utf-8-sig')

y_true_gk = oof['y_true'].values
assert (lasso['y_true'].values == y_true_gk).all()
assert (optuna['y_true'].values == y_true_gk).all()
assert (stack['y_true'].values == y_true_gk).all()
print(f"GroupKFold 样本量：{len(y_true_gk)}，舞弊率：{y_true_gk.mean():.4f}")

# ============================================================
# 2. 读取 TimeSplit Test
# ============================================================
ts = pd.read_csv(os.path.join(BASE, "TimeSplit_test_predictions.csv"), encoding='utf-8-sig')
print(f"\nTimeSplit 列名：{ts.columns.tolist()}")
print(f"TimeSplit shape：{ts.shape}")

# 自动识别列
y_true_ts = ts['y_true'].values if 'y_true' in ts.columns else ts.iloc[:, 0].values
prob_col_raw = None
prob_col_cal = None
for c in ['prob_raw', 'y_prob_raw', 'y_prob', 'prob']:
    if c in ts.columns:
        prob_col_raw = c
        break
for c in ['prob_cal', 'y_prob_calibrated', 'y_prob_cal', 'prob_calibrated']:
    if c in ts.columns:
        prob_col_cal = c
        break

print(f"TimeSplit raw 概率列：{prob_col_raw}")
print(f"TimeSplit cal 概率列：{prob_col_cal}")
print(f"TimeSplit 样本量：{len(y_true_ts)}，舞弊率：{y_true_ts.mean():.4f}")

# ============================================================
# 3. 评估函数
# ============================================================
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f1, best_th = 0, 0.5
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th

def eval_prob(y_true, y_prob, model_name, eval_type):
    auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    best_th = find_best_threshold(y_true, y_prob)
    y_pred = (y_prob >= best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'Model': model_name,
        'Evaluation': eval_type,
        'AUC': round(auc, 4),
        'PR_AUC': round(pr_auc, 4),
        'Brier': round(brier, 4),
        'Recall': round(tp/(tp+fn) if (tp+fn)>0 else 0, 4),
        'Precision': round(tp/(tp+fp) if (tp+fp)>0 else 0, 4),
        'Type_I_Error': round(fp/(fp+tn) if (fp+tn)>0 else 0, 4),
        'Type_II_Error': round(fn/(fn+tp) if (fn+tp)>0 else 0, 4),
        'Threshold': round(best_th, 3),
        'N': len(y_true),
        'Fraud_rate': round(y_true.mean(), 4)
    }

# ============================================================
# 4. GroupKFold 各行
# ============================================================
rows = []
rows.append(eval_prob(y_true_gk, lasso['lasso_full'].values, 'Lasso-LR', 'GroupKFold'))
rows.append(eval_prob(y_true_gk, oof['Full_RandomForest'].values, 'Random Forest', 'GroupKFold'))
rows.append(eval_prob(y_true_gk, oof['Full_XGBoost'].values, 'XGBoost', 'GroupKFold'))
rows.append(eval_prob(y_true_gk, oof['Full_LightGBM'].values, 'LightGBM (default)', 'GroupKFold'))
rows.append(eval_prob(y_true_gk, optuna['y_prob_raw'].values, 'LightGBM + Optuna', 'GroupKFold'))

row_cal = eval_prob(y_true_gk, optuna['y_prob_calibrated'].values,
                     'LightGBM + Optuna + Cal', 'GroupKFold')
row_cal['AUC_raw'] = round(roc_auc_score(y_true_gk, optuna['y_prob_raw'].values), 4)
row_cal['Brier_raw'] = round(brier_score_loss(y_true_gk, optuna['y_prob_raw'].values), 4)
rows.append(row_cal)

rows.append(eval_prob(y_true_gk, stack['stack'].values, 'Stacking', 'GroupKFold'))

# ============================================================
# 5. TimeSplit 主结果
# ============================================================
if prob_col_cal is not None:
    row_ts = eval_prob(y_true_ts, ts[prob_col_cal].values,
                        'LightGBM + Optuna + Cal', 'TimeSplit')
    if prob_col_raw is not None:
        row_ts['AUC_raw'] = round(roc_auc_score(y_true_ts, ts[prob_col_raw].values), 4)
        row_ts['Brier_raw'] = round(brier_score_loss(y_true_ts, ts[prob_col_raw].values), 4)
    rows.append(row_ts)
elif prob_col_raw is not None:
    row_ts = eval_prob(y_true_ts, ts[prob_col_raw].values,
                        'LightGBM + Optuna + Cal', 'TimeSplit')
    rows.append(row_ts)

# ============================================================
# 6. 输出
# ============================================================
table3 = pd.DataFrame(rows)

col_order = ['Model', 'Evaluation', 'AUC', 'AUC_raw', 'PR_AUC',
             'Brier', 'Brier_raw', 'Recall', 'Precision',
             'Type_I_Error', 'Type_II_Error', 'Threshold', 'N', 'Fraud_rate']
col_order = [c for c in col_order if c in table3.columns]
table3 = table3[col_order]

table3.to_csv(os.path.join(OUT, "Table3_MainResults.csv"),
              index=False, encoding='utf-8-sig')

print("\n" + "=" * 110)
print("Table 3 主结果表：")
print("=" * 110)
print(table3.to_string(index=False))

GroupKFold 样本量：48328，舞弊率：0.0876

TimeSplit 列名：['y_true', 'prob_raw', 'prob_cal']
TimeSplit shape：(11237, 3)
TimeSplit raw 概率列：prob_raw
TimeSplit cal 概率列：prob_cal
TimeSplit 样本量：11237，舞弊率：0.0815

Table 3 主结果表：
                  Model Evaluation    AUC  AUC_raw  PR_AUC  Brier  Brier_raw  Recall  Precision  Type_I_Error  Type_II_Error  Threshold     N  Fraud_rate
               Lasso-LR GroupKFold 0.7379      NaN  0.2308 0.2028        NaN  0.4035     0.2309        0.1290         0.5965       0.62 48328      0.0876
          Random Forest GroupKFold 0.7819      NaN  0.2711 0.0898        NaN  0.5011     0.2541        0.1412         0.4989       0.33 48328      0.0876
                XGBoost GroupKFold 0.7679      NaN  0.2801 0.0997        NaN  0.3891     0.2814        0.0954         0.6109       0.48 48328      0.0876
     LightGBM (default) GroupKFold 0.7753      NaN  0.2897 0.1099        NaN  0.4021     0.2878        0.0955         0.5979       0.53 48328      0.0876
      LightGBM + Optun

In [4]:
# -*- coding: utf-8 -*-
"""
重跑 TimeSplit，保存与 4 个稳健性实验口径一致的预测文件
输出：
  TimeSplit_test_predictions_FINAL.csv
  TimeSplit_val_predictions_FINAL.csv
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix, f1_score
from sklearn.isotonic import IsotonicRegression

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

# ============================================================
# 1. 读取建模数据（与稳健性实验一致）
# ============================================================
df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_未隔离.csv"), encoding='utf-8-sig')
print(f"总样本：{len(df)}")

# 特征列
exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

# one-hot object 列
obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"特征数：{len(feature_cols)}")

# ============================================================
# 2. 时间外推划分
# ============================================================
train = df[df['year'] <= 2021]
val   = df[df['year'] == 2022]
test  = df[df['year'] >= 2023]

print(f"Train: {len(train)} (fraud rate {train['Fraud'].mean():.4f})")
print(f"Val: {len(val)} (fraud rate {val['Fraud'].mean():.4f})")
print(f"Test: {len(test)} (fraud rate {test['Fraud'].mean():.4f})")

X_train, y_train = train[feature_cols], train['Fraud'].values
X_val,   y_val   = val[feature_cols],   val['Fraud'].values
X_test,  y_test  = test[feature_cols],  test['Fraud'].values

# ============================================================
# 3. 主模型参数（与 4 个稳健性实验一致）
# ============================================================
params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

print("\n训练 LightGBM ...")
model = lgb.LGBMClassifier(**params)
model.fit(X_train, y_train)

# ============================================================
# 4. Isotonic 校准（在 Val 上拟合）
# ============================================================
prob_val_raw = model.predict_proba(X_val)[:, 1]
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(prob_val_raw, y_val)
prob_val_cal = iso.predict(prob_val_raw)

prob_test_raw = model.predict_proba(X_test)[:, 1]
prob_test_cal = iso.predict(prob_test_raw)

# ============================================================
# 5. 评估
# ============================================================
auc_raw = roc_auc_score(y_test, prob_test_raw)
auc_cal = roc_auc_score(y_test, prob_test_cal)
pr_auc_cal = average_precision_score(y_test, prob_test_cal)
brier_raw = brier_score_loss(y_test, prob_test_raw)
brier_cal = brier_score_loss(y_test, prob_test_cal)

print(f"\nTimeSplit Test:")
print(f"  AUC raw = {auc_raw:.4f}")
print(f"  AUC cal = {auc_cal:.4f}")
print(f"  PR-AUC = {pr_auc_cal:.4f}")
print(f"  Brier raw = {brier_raw:.4f}")
print(f"  Brier cal = {brier_cal:.4f}")

# 验证集选阈值（F1 最优）
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f1, best_th = 0, 0.5
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th

best_th = find_best_threshold(y_val, prob_val_cal)
print(f"  Best threshold (val) = {best_th:.2f}")

y_pred = (prob_test_cal >= best_th).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
print(f"  Recall = {tp/(tp+fn):.4f}")
print(f"  Precision = {tp/(tp+fp):.4f}")
print(f"  Type II = {fn/(fn+tp):.4f}")

# ============================================================
# 6. 保存预测
# ============================================================
test_pred = pd.DataFrame({
    'y_true': y_test,
    'prob_raw': prob_test_raw,
    'prob_cal': prob_test_cal
})
test_pred.to_csv(os.path.join(OUT, "TimeSplit_test_predictions_FINAL.csv"),
                  index=False, encoding='utf-8-sig')

val_pred = pd.DataFrame({
    'y_true': y_val,
    'prob_raw': prob_val_raw,
    'prob_cal': prob_val_cal
})
val_pred.to_csv(os.path.join(OUT, "TimeSplit_val_predictions_FINAL.csv"),
                 index=False, encoding='utf-8-sig')

print("\n完成！输出：")
print("  TimeSplit_test_predictions_FINAL.csv")
print("  TimeSplit_val_predictions_FINAL.csv")

总样本：48328
特征数：89
Train: 31553 (fraud rate 0.0884)
Val: 5538 (fraud rate 0.0953)
Test: 11237 (fraud rate 0.0815)

训练 LightGBM ...

TimeSplit Test:
  AUC raw = 0.7668
  AUC cal = 0.7642
  PR-AUC = 0.2489
  Brier raw = 0.2086
  Brier cal = 0.0684
  Best threshold (val) = 0.17
  Recall = 0.5142
  Precision = 0.2307
  Type II = 0.4858

完成！输出：
  TimeSplit_test_predictions_FINAL.csv
  TimeSplit_val_predictions_FINAL.csv


In [5]:
import os
import pandas as pd

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

# 读取 TimeSplit_Bootstrap_CI.csv，看是否有更多信息
ci = pd.read_excel(os.path.join(BASE, "TimeSplit_Bootstrap_CI.xlsx"))
print("Bootstrap CI 列名：", ci.columns.tolist())
print(ci)

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\科研\\二次实验\\二次实验\\二次实验\\国泰安数据下载\\合并结果\\TimeSplit_Bootstrap_CI.xlsx'

In [6]:
import os
import pandas as pd

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

# 搜索所有 Bootstrap 相关文件
print("=" * 70)
print("Bootstrap 相关文件：")
print("=" * 70)
for f in os.listdir(BASE):
    if 'bootstrap' in f.lower() or 'Bootstrap' in f:
        path = os.path.join(BASE, f)
        size = os.path.getsize(path) / 1024
        print(f"{f}  ({size:.1f} KB)")

# 读取 TimeSplit_Bootstrap_CI.csv
ci_path = os.path.join(BASE, "TimeSplit_Bootstrap_CI.csv")
if os.path.exists(ci_path):
    ci = pd.read_csv(ci_path, encoding='utf-8-sig')
    print("\n" + "=" * 70)
    print("TimeSplit_Bootstrap_CI.csv 内容：")
    print("=" * 70)
    print(f"列名：{ci.columns.tolist()}")
    print(ci.to_string(index=False))
else:
    print("\n未找到 TimeSplit_Bootstrap_CI.csv")

Bootstrap 相关文件：
TimeSplit_Bootstrap_CI.csv  (0.6 KB)

TimeSplit_Bootstrap_CI.csv 内容：
列名：['Set', 'Version', 'Metric', 'Mean', 'CI_low', 'CI_high']
           Set    Version Metric     Mean   CI_low  CI_high
TimeSplit-Test        Raw    AUC 0.757590 0.741239 0.773660
TimeSplit-Test        Raw PR-AUC 0.251543 0.224904 0.279456
TimeSplit-Test        Raw  Brier 0.211204 0.207404 0.214857
TimeSplit-Test Calibrated    AUC 0.756088 0.739919 0.772825
TimeSplit-Test Calibrated PR-AUC 0.234302 0.209523 0.260488
TimeSplit-Test Calibrated  Brier 0.069286 0.065853 0.072741


In [7]:
# -*- coding: utf-8 -*-
"""
重跑 GroupKFold Bootstrap CI
- 用 Optuna_calibrated_predictions.csv（与 Table 3 同源）
- 同时报告校准后和 raw 的 CI
输出：
  Bootstrap_CI_GroupKFold_FINAL.csv
"""
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

optuna = pd.read_csv(os.path.join(BASE, "Optuna_calibrated_predictions.csv"), encoding='utf-8-sig')
y_true = optuna['y_true'].values
y_prob_cal = optuna['y_prob_calibrated'].values
y_prob_raw = optuna['y_prob_raw'].values

# 点估计
auc_cal = roc_auc_score(y_true, y_prob_cal)
auc_raw = roc_auc_score(y_true, y_prob_raw)
pr_cal = average_precision_score(y_true, y_prob_cal)
brier_cal = brier_score_loss(y_true, y_prob_cal)
brier_raw = brier_score_loss(y_true, y_prob_raw)

print(f"点估计 (calibrated): AUC = {auc_cal:.4f}")
print(f"点估计 (raw):        AUC = {auc_raw:.4f}")
print(f"点估计 PR-AUC (cal): {pr_cal:.4f}")
print(f"点估计 Brier (cal):  {brier_cal:.4f}")
print(f"点估计 Brier (raw):  {brier_raw:.4f}")

# Bootstrap
np.random.seed(42)
n = len(y_true)
n_boot = 1000

aucs_cal, aucs_raw, prs_cal, briers_cal, briers_raw = [], [], [], [], []

for i in range(n_boot):
    idx = np.random.choice(n, n, replace=True)
    y_b = y_true[idx]
    if y_b.sum() == 0 or y_b.sum() == len(y_b):
        continue
    p_cal = y_prob_cal[idx]
    p_raw = y_prob_raw[idx]
    aucs_cal.append(roc_auc_score(y_b, p_cal))
    aucs_raw.append(roc_auc_score(y_b, p_raw))
    prs_cal.append(average_precision_score(y_b, p_cal))
    briers_cal.append(brier_score_loss(y_b, p_cal))
    briers_raw.append(brier_score_loss(y_b, p_raw))

def ci(arr):
    return np.percentile(arr, [2.5, 97.5])

result = pd.DataFrame([
    {'metric': 'AUC (calibrated)', 'point_estimate': round(auc_cal, 4),
     'CI_lower': round(ci(aucs_cal)[0], 4), 'CI_upper': round(ci(aucs_cal)[1], 4)},
    {'metric': 'AUC (raw)', 'point_estimate': round(auc_raw, 4),
     'CI_lower': round(ci(aucs_raw)[0], 4), 'CI_upper': round(ci(aucs_raw)[1], 4)},
    {'metric': 'PR-AUC (calibrated)', 'point_estimate': round(pr_cal, 4),
     'CI_lower': round(ci(prs_cal)[0], 4), 'CI_upper': round(ci(prs_cal)[1], 4)},
    {'metric': 'Brier (calibrated)', 'point_estimate': round(brier_cal, 4),
     'CI_lower': round(ci(briers_cal)[0], 4), 'CI_upper': round(ci(briers_cal)[1], 4)},
    {'metric': 'Brier (raw)', 'point_estimate': round(brier_raw, 4),
     'CI_lower': round(ci(briers_raw)[0], 4), 'CI_upper': round(ci(briers_raw)[1], 4)},
])

result.to_csv(os.path.join(OUT, "Bootstrap_CI_GroupKFold_FINAL.csv"),
              index=False, encoding='utf-8-sig')

print("\n" + "=" * 80)
print("Bootstrap 95% CI（GroupKFold OOF，用 Table 3 同源文件）")
print("=" * 80)
print(result.to_string(index=False))

点估计 (calibrated): AUC = 0.7873
点估计 (raw):        AUC = 0.7881
点估计 PR-AUC (cal): 0.2986
点估计 Brier (cal):  0.0704
点估计 Brier (raw):  0.1617

Bootstrap 95% CI（GroupKFold OOF，用 Table 3 同源文件）
             metric  point_estimate  CI_lower  CI_upper
   AUC (calibrated)          0.7873    0.7809    0.7938
          AUC (raw)          0.7881    0.7814    0.7946
PR-AUC (calibrated)          0.2986    0.2851    0.3134
 Brier (calibrated)          0.0704    0.0688    0.0723
        Brier (raw)          0.1617    0.1600    0.1632


In [8]:
# -*- coding: utf-8 -*-
"""
重跑 TimeSplit Bootstrap CI
- 用 TimeSplit_test_predictions_FINAL.csv（与最新主结果同源）
输出：
  Bootstrap_CI_TimeSplit_FINAL.csv
"""
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

ts = pd.read_csv(os.path.join(BASE, "TimeSplit_test_predictions_FINAL.csv"), encoding='utf-8-sig')
y_true = ts['y_true'].values
y_prob_cal = ts['prob_cal'].values
y_prob_raw = ts['prob_raw'].values

auc_cal = roc_auc_score(y_true, y_prob_cal)
auc_raw = roc_auc_score(y_true, y_prob_raw)
pr_cal = average_precision_score(y_true, y_prob_cal)
brier_cal = brier_score_loss(y_true, y_prob_cal)
brier_raw = brier_score_loss(y_true, y_prob_raw)

print(f"TimeSplit 点估计 (calibrated): AUC = {auc_cal:.4f}")
print(f"TimeSplit 点估计 (raw):        AUC = {auc_raw:.4f}")

np.random.seed(42)
n = len(y_true)
n_boot = 1000

aucs_cal, aucs_raw, prs_cal, briers_cal, briers_raw = [], [], [], [], []

for i in range(n_boot):
    idx = np.random.choice(n, n, replace=True)
    y_b = y_true[idx]
    if y_b.sum() == 0 or y_b.sum() == len(y_b):
        continue
    p_cal = y_prob_cal[idx]
    p_raw = y_prob_raw[idx]
    aucs_cal.append(roc_auc_score(y_b, p_cal))
    aucs_raw.append(roc_auc_score(y_b, p_raw))
    prs_cal.append(average_precision_score(y_b, p_cal))
    briers_cal.append(brier_score_loss(y_b, p_cal))
    briers_raw.append(brier_score_loss(y_b, p_raw))

def ci(arr):
    return np.percentile(arr, [2.5, 97.5])

result = pd.DataFrame([
    {'metric': 'AUC (calibrated)', 'point_estimate': round(auc_cal, 4),
     'CI_lower': round(ci(aucs_cal)[0], 4), 'CI_upper': round(ci(aucs_cal)[1], 4)},
    {'metric': 'AUC (raw)', 'point_estimate': round(auc_raw, 4),
     'CI_lower': round(ci(aucs_raw)[0], 4), 'CI_upper': round(ci(aucs_raw)[1], 4)},
    {'metric': 'PR-AUC (calibrated)', 'point_estimate': round(pr_cal, 4),
     'CI_lower': round(ci(prs_cal)[0], 4), 'CI_upper': round(ci(prs_cal)[1], 4)},
    {'metric': 'Brier (calibrated)', 'point_estimate': round(brier_cal, 4),
     'CI_lower': round(ci(briers_cal)[0], 4), 'CI_upper': round(ci(briers_cal)[1], 4)},
    {'metric': 'Brier (raw)', 'point_estimate': round(brier_raw, 4),
     'CI_lower': round(ci(briers_raw)[0], 4), 'CI_upper': round(ci(briers_raw)[1], 4)},
])

result.to_csv(os.path.join(OUT, "Bootstrap_CI_TimeSplit_FINAL.csv"),
              index=False, encoding='utf-8-sig')

print("\n" + "=" * 80)
print("Bootstrap 95% CI（TimeSplit，用最新 FINAL 预测文件）")
print("=" * 80)
print(result.to_string(index=False))

TimeSplit 点估计 (calibrated): AUC = 0.7642
TimeSplit 点估计 (raw):        AUC = 0.7668

Bootstrap 95% CI（TimeSplit，用最新 FINAL 预测文件）
             metric  point_estimate  CI_lower  CI_upper
   AUC (calibrated)          0.7642    0.7476    0.7821
          AUC (raw)          0.7668    0.7502    0.7842
PR-AUC (calibrated)          0.2489    0.2252    0.2773
 Brier (calibrated)          0.0684    0.0652    0.0718
        Brier (raw)          0.2086    0.2048    0.2124


In [9]:
# -*- coding: utf-8 -*-
"""
批量重命名 7 张图：
  fig1_roc → fig2_roc
  fig2_pr → fig3_pr
  fig3_calibration → fig4_calibration
  fig4_shap_bar → fig5_shap_bar
  fig5_shap_beeswarm → fig6_shap_beeswarm
  fig6_shap_dependence → fig7_shap_dependence
  fig7_shap_stability → fig8_shap_stability

fig8_smote_vs_classweight_shap 保持不变（移到附录作 A1）

每个图有 .png 和 .pdf 两个文件，都重命名。
"""
import os
import shutil

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表"

# 映射关系（旧名 → 新名）
rename_map = {
    'fig1_roc': 'fig2_roc',
    'fig2_pr': 'fig3_pr',
    'fig3_calibration': 'fig4_calibration',
    'fig4_shap_bar': 'fig5_shap_bar',
    'fig5_shap_beeswarm': 'fig6_shap_beeswarm',
    'fig6_shap_dependence': 'fig7_shap_dependence',
    'fig7_shap_stability': 'fig8_shap_stability',
}

# 注意：需要按顺序倒序处理，避免覆盖
# 比如 fig2_pr → fig3_pr 之前，要先处理 fig3 → fig4
ordered = sorted(rename_map.keys(), key=lambda x: -int(x.split('_')[0].replace('fig', '')))

# 先备份一份到 backup 目录
backup_dir = os.path.join(BASE, "backup_before_rename")
if not os.path.exists(backup_dir):
    os.makedirs(backup_dir)
    print(f"已创建备份目录：{backup_dir}")

# 备份全部图
for f in os.listdir(BASE):
    if f.startswith('fig') and (f.endswith('.png') or f.endswith('.pdf')):
        shutil.copy2(os.path.join(BASE, f), os.path.join(backup_dir, f))
print("已备份所有 fig* 文件到 backup_before_rename/")

# 分两步重命名，避免冲突
# 第一步：把 fig1_roc → tmp_fig2_roc，fig2_pr → tmp_fig3_pr，...
print("\n第一步：临时重命名")
temp_map = {}
for old_prefix, new_prefix in rename_map.items():
    for ext in ['.png', '.pdf']:
        old_file = os.path.join(BASE, old_prefix + ext)
        if os.path.exists(old_file):
            temp_name = 'tmp_' + new_prefix + ext
            temp_path = os.path.join(BASE, temp_name)
            os.rename(old_file, temp_path)
            temp_map[temp_path] = os.path.join(BASE, new_prefix + ext)
            print(f"  {old_prefix + ext} → {temp_name}")
        else:
            print(f"  [跳过] {old_prefix + ext} 不存在")

# 第二步：tmp_fig2_roc → fig2_roc
print("\n第二步：最终重命名")
for temp_path, final_path in temp_map.items():
    os.rename(temp_path, final_path)
    print(f"  {os.path.basename(temp_path)} → {os.path.basename(final_path)}")

# 验证
print("\n" + "=" * 60)
print("重命名后目录内容：")
print("=" * 60)
for f in sorted(os.listdir(BASE)):
    if f.startswith('fig') and (f.endswith('.png') or f.endswith('.pdf')):
        size = os.path.getsize(os.path.join(BASE, f)) / 1024
        print(f"  {f}  ({size:.1f} KB)")

print("\n注意：fig8_smote_vs_classweight_shap 保持原名，作为附录图 A1")

已创建备份目录：D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\backup_before_rename
已备份所有 fig* 文件到 backup_before_rename/

第一步：临时重命名
  fig1_roc.png → tmp_fig2_roc.png
  fig1_roc.pdf → tmp_fig2_roc.pdf
  fig2_pr.png → tmp_fig3_pr.png
  fig2_pr.pdf → tmp_fig3_pr.pdf
  fig3_calibration.png → tmp_fig4_calibration.png
  fig3_calibration.pdf → tmp_fig4_calibration.pdf
  fig4_shap_bar.png → tmp_fig5_shap_bar.png
  fig4_shap_bar.pdf → tmp_fig5_shap_bar.pdf
  fig5_shap_beeswarm.png → tmp_fig6_shap_beeswarm.png
  fig5_shap_beeswarm.pdf → tmp_fig6_shap_beeswarm.pdf
  fig6_shap_dependence.png → tmp_fig7_shap_dependence.png
  fig6_shap_dependence.pdf → tmp_fig7_shap_dependence.pdf
  fig7_shap_stability.png → tmp_fig8_shap_stability.png
  fig7_shap_stability.pdf → tmp_fig8_shap_stability.pdf

第二步：最终重命名
  tmp_fig2_roc.png → fig2_roc.png
  tmp_fig2_roc.pdf → fig2_roc.pdf
  tmp_fig3_pr.png → fig3_pr.png
  tmp_fig3_pr.pdf → fig3_pr.pdf
  tmp_fig4_calibration.png → fig4_calibration.png
  tmp_fig4_calibration.pdf → fig4_calibr

In [10]:
# -*- coding: utf-8 -*-
"""
补画附录图 A2：行业分层 ROC
- 对比 Full sample / Manufacturing / Non-Manufacturing 在 Test 2023–2024 上的 ROC 曲线
- 使用 Industry_Stratification_Calibrated.csv 里的数字（已是校准后 AUC）
- 为了画 ROC 曲线，需要重新训练并保存概率
输出：
  figA2_industry_roc.png
  figA2_industry_roc.pdf
"""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.isotonic import IsotonicRegression

plt.rcParams['font.sans-serif'] = ['Arial', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT_DIR = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表"
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

# 读取带行业信息的建模数据
df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_行业.csv"), encoding='utf-8-sig')
df = df[df['IndustryName'].notna()].copy()
df['is_manufacturing'] = df['IndustryName'].str.contains('制造业', na=False).astype(int)

# 特征列
exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1', 'IndustryName',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set',
                'is_manufacturing']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]

params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def get_roc_data(sub_df, label):
    train = sub_df[sub_df['year'] <= 2021]
    val = sub_df[sub_df['year'] == 2022]
    test = sub_df[sub_df['year'] >= 2023]

    X_train, y_train = train[feature_cols], train['Fraud'].values
    X_val, y_val = val[feature_cols], val['Fraud'].values
    X_test, y_test = test[feature_cols], test['Fraud'].values

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    prob_val = model.predict_proba(X_val)[:, 1]
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(prob_val, y_val)

    prob_test = model.predict_proba(X_test)[:, 1]
    prob_test_cal = iso.predict(prob_test)

    fpr, tpr, _ = roc_curve(y_test, prob_test_cal)
    auc = roc_auc_score(y_test, prob_test_cal)
    return fpr, tpr, auc, label

# 三个分组
results = {}
results['Full sample'] = get_roc_data(df, 'Full sample')
results['Manufacturing'] = get_roc_data(df[df['is_manufacturing'] == 1], 'Manufacturing')
results['Non-Manufacturing'] = get_roc_data(df[df['is_manufacturing'] == 0], 'Non-Manufacturing')

# 画图
plt.figure(figsize=(7, 6))
colors = {'Full sample': 'tab:blue', 'Manufacturing': 'tab:orange', 'Non-Manufacturing': 'tab:green'}
for label, (fpr, tpr, auc, _) in results.items():
    plt.plot(fpr, tpr, color=colors[label], lw=2,
             label=f'{label} (AUC = {auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves by Industry Category (Time Split Test 2023–2024)', fontsize=13)
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(os.path.join(OUT_DIR, "figA2_industry_roc.png"), dpi=300)
plt.savefig(os.path.join(OUT_DIR, "figA2_industry_roc.pdf"))
plt.close()
print("完成：figA2_industry_roc.png / .pdf")

完成：figA2_industry_roc.png / .pdf


In [11]:
# -*- coding: utf-8 -*-
"""
补画附录图 A3：窄口径（P2501+P2502）ROC 曲线
- 对比宽口径（P2501+P2502+P2503+P2506）和窄口径的 ROC
输出：
  figA3_narrow_fraud_roc.png
  figA3_narrow_fraud_roc.pdf
"""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
from sklearn.isotonic import IsotonicRegression

plt.rcParams['font.sans-serif'] = ['Arial', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT_DIR = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表"

# 读取建模数据
df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_未隔离.csv"), encoding='utf-8-sig')

# 读取违规明细，构建窄口径标签
viol = pd.read_csv(os.path.join(BASE, "违规目标四类_公司年度类型.csv"), encoding='utf-8-sig')

def norm_stkcd(s):
    return s.astype(str).str.replace(r'\.0$', '', regex=True).str.strip().str.zfill(6)

viol['Stkcd'] = norm_stkcd(viol['Stkcd'])
df['Stkcd'] = norm_stkcd(df['Stkcd'])

narrow = viol[viol['ViolationTypeID'].isin(['P2501', 'P2502'])]
narrow_keys = narrow[['Stkcd', 'year']].drop_duplicates()
narrow_keys['Fraud_narrow'] = 1

df = df.merge(narrow_keys, on=['Stkcd', 'year'], how='left')
df['Fraud_narrow'] = df['Fraud_narrow'].fillna(0).astype(int)

# 特征列
exclude_cols = ['Stkcd', 'year', 'Fraud', 'Fraud_narrow', 'ShortName', 'IndustryName1',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]

params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def get_roc_data(sub_df, label_col, label):
    train = sub_df[sub_df['year'] <= 2021]
    val = sub_df[sub_df['year'] == 2022]
    test = sub_df[sub_df['year'] >= 2023]

    X_train, y_train = train[feature_cols], train[label_col].values
    X_val, y_val = val[feature_cols], val[label_col].values
    X_test, y_test = test[feature_cols], test[label_col].values

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    prob_val = model.predict_proba(X_val)[:, 1]
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(prob_val, y_val)

    prob_test = model.predict_proba(X_test)[:, 1]
    prob_test_cal = iso.predict(prob_test)

    fpr, tpr, _ = roc_curve(y_test, prob_test_cal)
    auc = roc_auc_score(y_test, prob_test_cal)
    pr_prec, pr_rec, _ = precision_recall_curve(y_test, prob_test_cal)
    pr_auc = average_precision_score(y_test, prob_test_cal)
    return fpr, tpr, auc, pr_prec, pr_rec, pr_auc, label

# 两种标签
res_wide = get_roc_data(df, 'Fraud', 'Wide (P2501+P2502+P2503+P2506)')
res_narrow = get_roc_data(df, 'Fraud_narrow', 'Narrow (P2501+P2502)')

# 画双面板：左 ROC，右 PR
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左：ROC
for res, color in [(res_wide, 'tab:blue'), (res_narrow, 'tab:red')]:
    fpr, tpr, auc, _, _, _, label = res
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC = {auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves: Wide vs Narrow Fraud Definition', fontsize=12)
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(alpha=0.3)

# 右：PR
for res, color in [(res_wide, 'tab:blue'), (res_narrow, 'tab:red')]:
    _, _, _, pr_prec, pr_rec, pr_auc, label = res
    axes[1].plot(pr_rec, pr_prec, color=color, lw=2, label=f'{label} (PR-AUC = {pr_auc:.4f})')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curves: Wide vs Narrow Fraud Definition', fontsize=12)
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figA3_narrow_fraud_roc.png"), dpi=300)
plt.savefig(os.path.join(OUT_DIR, "figA3_narrow_fraud_roc.pdf"))
plt.close()
print("完成：figA3_narrow_fraud_roc.png / .pdf")

# 打印数字确认
print(f"\n宽口径：AUC = {res_wide[2]:.4f}, PR-AUC = {res_wide[5]:.4f}")
print(f"窄口径：AUC = {res_narrow[2]:.4f}, PR-AUC = {res_narrow[5]:.4f}")

完成：figA3_narrow_fraud_roc.png / .pdf

宽口径：AUC = 0.7642, PR-AUC = 0.2489
窄口径：AUC = 0.7989, PR-AUC = 0.0376


In [1]:
# -*- coding: utf-8 -*-
"""
重画 ROC 曲线（更新版）：
- 特征数改为 84
- 4 个模型 × 2 特征集（Base=57 / Full=84）
- 使用与主实验一致的 OOF 预测数据
输出：
  fig2_roc.png / .pdf（新编号）
"""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

plt.rcParams['font.sans-serif'] = ['Arial', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT_DIR = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表"

# ============================================================
# 读取 OOF 预测
# ============================================================
oof = pd.read_csv(os.path.join(BASE, "对比_OOF_predictions.csv"), encoding='utf-8-sig')
lasso = pd.read_csv(os.path.join(BASE, "LassoLR_OOF.csv"), encoding='utf-8-sig')

y_true = oof['y_true'].values

# ============================================================
# 计算 4 模型 × 2 特征集 的 ROC
# ============================================================
# Base 特征集（57 个特征）：Lasso / LGB / XGB / RF
# Full 特征集（84 个特征）：Lasso / LGB / XGB / RF

def get_roc(y_true, y_prob, label):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    return fpr, tpr, auc, label

# Base 特征集
base_curves = [
    get_roc(y_true, lasso['lasso_base'].values, 'Lasso-LR'),
    get_roc(y_true, oof['Base_LightGBM'].values, 'LightGBM'),
    get_roc(y_true, oof['Base_XGBoost'].values, 'XGBoost'),
    get_roc(y_true, oof['Base_RandomForest'].values, 'Random Forest'),
]

# Full 特征集
full_curves = [
    get_roc(y_true, lasso['lasso_full'].values, 'Lasso-LR'),
    get_roc(y_true, oof['Full_LightGBM'].values, 'LightGBM'),
    get_roc(y_true, oof['Full_XGBoost'].values, 'XGBoost'),
    get_roc(y_true, oof['Full_RandomForest'].values, 'Random Forest'),
]

# ============================================================
# 画双面板 ROC
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = {
    'Lasso-LR': 'tab:red',
    'LightGBM': 'tab:blue',
    'XGBoost': 'tab:orange',
    'Random Forest': 'tab:green',
}

# 左：Base 特征集（57 特征）
for fpr, tpr, auc, label in base_curves:
    axes[0].plot(fpr, tpr, color=colors[label], lw=2,
                 label=f'{label} (AUC = {auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('Base feature set (57 features)', fontsize=12)
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(alpha=0.3)

# 右：Full 特征集（84 特征）
for fpr, tpr, auc, label in full_curves:
    axes[1].plot(fpr, tpr, color=colors[label], lw=2,
                 label=f'{label} (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('Full feature set (84 features)', fontsize=12)
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig2_roc.png"), dpi=300)
plt.savefig(os.path.join(OUT_DIR, "fig2_roc.pdf"))
plt.close()
print("完成：fig2_roc.png / .pdf")

# 打印 AUC 汇总
print("\nBase 特征集 AUC：")
for _, _, auc, label in base_curves:
    print(f"  {label}: {auc:.4f}")
print("\nFull 特征集 AUC：")
for _, _, auc, label in full_curves:
    print(f"  {label}: {auc:.4f}")

完成：fig2_roc.png / .pdf

Base 特征集 AUC：
  Lasso-LR: 0.7265
  LightGBM: 0.7747
  XGBoost: 0.7656
  Random Forest: 0.7810

Full 特征集 AUC：
  Lasso-LR: 0.7379
  LightGBM: 0.7753
  XGBoost: 0.7679
  Random Forest: 0.7819


In [2]:
# -*- coding: utf-8 -*-
"""
重画 PR 曲线（更新版）
输出：
  fig3_pr.png / .pdf
"""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.rcParams['font.sans-serif'] = ['Arial', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT_DIR = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表"

oof = pd.read_csv(os.path.join(BASE, "对比_OOF_predictions.csv"), encoding='utf-8-sig')
lasso = pd.read_csv(os.path.join(BASE, "LassoLR_OOF.csv"), encoding='utf-8-sig')

y_true = oof['y_true'].values
baseline = y_true.mean()

def get_pr(y_true, y_prob, label):
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    return prec, rec, pr_auc, label

base_curves = [
    get_pr(y_true, lasso['lasso_base'].values, 'Lasso-LR'),
    get_pr(y_true, oof['Base_LightGBM'].values, 'LightGBM'),
    get_pr(y_true, oof['Base_XGBoost'].values, 'XGBoost'),
    get_pr(y_true, oof['Base_RandomForest'].values, 'Random Forest'),
]
full_curves = [
    get_pr(y_true, lasso['lasso_full'].values, 'Lasso-LR'),
    get_pr(y_true, oof['Full_LightGBM'].values, 'LightGBM'),
    get_pr(y_true, oof['Full_XGBoost'].values, 'XGBoost'),
    get_pr(y_true, oof['Full_RandomForest'].values, 'Random Forest'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = {'Lasso-LR': 'tab:red', 'LightGBM': 'tab:blue',
          'XGBoost': 'tab:orange', 'Random Forest': 'tab:green'}

for prec, rec, pr_auc, label in base_curves:
    axes[0].plot(rec, prec, color=colors[label], lw=2,
                 label=f'{label} (PR-AUC = {pr_auc:.4f})')
axes[0].axhline(baseline, color='gray', linestyle='--',
                label=f'Baseline ({baseline:.4f})')
axes[0].set_xlabel('Recall', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title('Base feature set (57 features)', fontsize=12)
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(alpha=0.3)

for prec, rec, pr_auc, label in full_curves:
    axes[1].plot(rec, prec, color=colors[label], lw=2,
                 label=f'{label} (PR-AUC = {pr_auc:.4f})')
axes[1].axhline(baseline, color='gray', linestyle='--',
                label=f'Baseline ({baseline:.4f})')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Full feature set (84 features)', fontsize=12)
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig3_pr.png"), dpi=300)
plt.savefig(os.path.join(OUT_DIR, "fig3_pr.pdf"))
plt.close()
print("完成：fig3_pr.png / .pdf")

完成：fig3_pr.png / .pdf
